# 📊 Experiment1: Comprehensive Benchmarking Analysis

**Objective**: Systematic evaluation and comparison of machine learning methods for credit risk prediction across multiple datasets with and without hyperparameter optimization.

---

## 📋 Overview

This notebook presents a comprehensive analysis of **Experiment1**, which evaluates selected methods from Experiment0 across two critical credit risk tasks:

### Tasks
1. **PD (Probability of Default)**: Binary classification predicting borrower default
2. **LGD (Loss Given Default)**: Regression estimating loss severity upon default

### Experimental Design
Each method is evaluated in two configurations:
- **NO_HPO**: Default hyperparameters (baseline)
- **HPO**: Hyperparameters optimized via Optuna (tuned)

Multiple datasets are used for each task with 5-fold cross-validation, ensuring robust performance estimates.

---

## 🔬 Analysis Components

This notebook provides four main analysis perspectives:

### Part A: PD Task Analysis (Classification)
- **A1. Performance Heatmaps**: Dataset × Method matrices showing AUC scores
- **A2. Performance Distributions**: Bar charts and boxplots across folds
- **A3. Rank Analysis**: Method rankings with heatmaps and average ranks
- **A4. PAMA Analysis**: Probability of Achieving Maximum AUC (fold-level)

### Part B: LGD Task Analysis (Regression)
- **B1. Performance Heatmaps**: Dataset × Method matrices showing R² scores
- **B2. Performance Distributions**: Bar charts and boxplots across folds
- **B3. Rank Analysis**: Method rankings with heatmaps and average ranks
- **B4. PAMA Analysis**: Probability of Achieving Maximum R² (fold-level)

### Part C: Dataset Characteristics Analysis
- **C1. Load Dataset Characteristics**: Extract size, features, dimensionality
- **C2. PD Rank Correlations**: Spearman correlations with dataset properties
- **C3. LGD Rank Correlations**: Identify which methods scale better

### Part D: Training Time Analysis
- **D1. PD Training Times**: Heatmaps and average times per method
- **D2. LGD Training Times**: Computational efficiency comparison
- **D3. HPO Impact on Time**: Additional computational cost of tuning

---

## 📊 Key Visualizations

For each task and analysis type, the notebook generates:
- **Heatmaps**: Color-coded matrices for easy pattern recognition
- **Bar Charts**: Average performance with error bars
- **Boxplots**: Distribution analysis across folds
- **Correlation Plots**: Method behavior vs dataset characteristics
- **PAMA Charts**: Win rate visualization

All visualizations are saved at 300 DPI in the `figures/` directory.

---

## 🎯 Research Questions Addressed

1. **Performance**: Which methods achieve the best predictive accuracy?
2. **HPO Impact**: How much does hyperparameter tuning improve results?
3. **Consistency**: Which methods perform reliably across datasets?
4. **Scalability**: How do methods perform on larger/more complex datasets?
5. **Efficiency**: What is the computational cost of each method?
6. **Specialization**: Do certain methods excel on specific dataset characteristics?

---

## 1. Setup & Configuration

In [ ]:
# Standard imports
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import friedmanchisquare, wilcoxon
import math

# =============================================================================
# SETUP PATHS (Notebook is in notebooks/ folder)
# =============================================================================

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent  # notebooks/ -> project root
sys.path.insert(0, str(PROJECT_ROOT))

# Import foundation methods list from method config
from src.methods.method_config import FOUNDATION_METHODS

EXPERIMENT_NAME = "experiment1"
RESULTS_DIR = PROJECT_ROOT / "results" / EXPERIMENT_NAME
SUMMARY_DIR = RESULTS_DIR / "summary"
FIGURES_DIR = RESULTS_DIR / "figures"

print("=" * 80)
print("  EXPERIMENT1 COMPREHENSIVE ANALYSIS")
print("=" * 80)
print(f"📂 Project root:       {PROJECT_ROOT}")
print(f"📂 Results directory:  {RESULTS_DIR}")
print(f"📂 Summary directory:  {SUMMARY_DIR}")
print(f"📂 Figures directory:  {FIGURES_DIR}")

# =============================================================================
# GENERATE SUMMARIES IF NOT ALREADY PRESENT
# =============================================================================

if SUMMARY_DIR.exists() and any(SUMMARY_DIR.glob("*.csv")):
    print(f"\n✓ Summary files already exist. Skipping generation.")
    print(f"   (Delete {SUMMARY_DIR} manually to regenerate)")
else:
    print(f"\n📊 Summary files not found. Generating summaries...")
    from src.utils.summarize_results import summarize_results
    print("\n" + "=" * 80)
    summarize_results(experiment=EXPERIMENT_NAME)
    print("=" * 80)

# Ensure figures directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("\n✅ Setup complete! Ready for analysis.")
print("=" * 80)

In [ ]:
# =============================================================================
# FONT SCALE CONFIGURATION
# =============================================================================
# Single configurable constant controlling all font sizes in the notebook.
# Adjust this value to scale ALL text sizes proportionally.
# Default: 1.0 (suitable for notebook viewing). Increase for paper figures.
FONT_SCALE = 1.5

# Plotting configuration (all sizes relative to FONT_SCALE)
plt.rcParams['figure.figsize'] = (20, 10)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = FONT_SCALE * 11
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.titlesize'] = FONT_SCALE * 15
plt.rcParams['axes.labelsize'] = FONT_SCALE * 12
plt.rcParams['xtick.labelsize'] = FONT_SCALE * 12
plt.rcParams['ytick.labelsize'] = FONT_SCALE * 10
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Derived font sizes (use these in plotting functions)
FS_TITLE = FONT_SCALE * 16       # Main plot titles
FS_SUBTITLE = FONT_SCALE * 14    # Subplot titles / secondary titles
FS_AXIS_LABEL = FONT_SCALE * 12  # Axis labels (xlabel, ylabel)
FS_TICK = FONT_SCALE * 9         # Tick labels
FS_TICK_Y = FONT_SCALE * 10      # Y-axis tick labels (slightly larger)
FS_ANNOT = FONT_SCALE * 7        # Heatmap annotation text
FS_LEGEND = FONT_SCALE * 10      # Legend text
FS_BAR_LABEL = FONT_SCALE * 9    # Labels on bar charts
FS_SUPTITLE = FONT_SCALE * 16    # Figure suptitles

# Color schemes
CMAP_PERFORMANCE = 'RdYlGn'  # Red (bad) to Green (good)
CMAP_TIME = 'YlOrRd'  # Yellow (fast) to Red (slow)

## 2. Data Loading

In [ ]:
# Load raw and aggregated results
pd_raw = pd.read_csv(SUMMARY_DIR / "summary_pd_raw.csv")
pd_agg = pd.read_csv(SUMMARY_DIR / "summary_pd_aggregated.csv")
lgd_raw = pd.read_csv(SUMMARY_DIR / "summary_lgd_raw.csv")
lgd_agg = pd.read_csv(SUMMARY_DIR / "summary_lgd_aggregated.csv")

print("\n📊 Data Overview:")
print(f"\nPD (Probability of Default):")
print(f"  - Raw results: {len(pd_raw)} fold results")
print(f"  - Methods: {pd_raw['method'].nunique()}")
print(f"  - Datasets: {pd_raw['dataset'].nunique()}")
print(f"  - HPO modes: {sorted(pd_raw['hpo_mode'].unique())}")
print(f"  - Methods: {sorted(pd_raw['method'].unique())}")

print(f"\nLGD (Loss Given Default):")
print(f"  - Raw results: {len(lgd_raw)} fold results")
print(f"  - Methods: {lgd_raw['method'].nunique()}")
print(f"  - Datasets: {lgd_raw['dataset'].nunique()}")
print(f"  - HPO modes: {sorted(lgd_raw['hpo_mode'].unique())}")
print(f"  - Methods: {sorted(lgd_raw['method'].unique())}")

## 3. Helper Functions

In [ ]:
def create_performance_heatmap(df, metric, hpo_mode, task_name, cmap='RdYlGn', figsize=(24, 12)):
    """
    Create a heatmap showing method performance across datasets.
    """
    mean_col = f'{metric}_mean'

    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found in data")
        return None, None

    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()

    if df_filtered.empty:
        print(f"Warning: No data for {hpo_mode}")
        return None, None

    pivot = df_filtered.pivot(index='dataset', columns='method', values=mean_col)
    pivot = pivot.sort_index()
    method_means = pivot.mean(axis=0).sort_values(ascending=False)
    pivot = pivot[method_means.index]

    # --- Vertical (default) ---
    fig, ax = plt.subplots(figsize=figsize)

    if metric == 'R2':
        vmin = pivot.min().min()
        vmax = pivot.max().max()
        abs_max = max(abs(vmin), abs(vmax))
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap='RdYlGn',
                    center=0, vmin=-abs_max, vmax=abs_max, annot_kws={'size': FS_ANNOT},
                    cbar_kws={'label': metric}, linewidths=0.5, ax=ax)
    else:
        vmin = pivot.min().min()
        vmax = pivot.max().max()
        sns.heatmap(pivot, annot=True, fmt='.3f', cmap=cmap,
                    vmin=vmin, vmax=vmax, annot_kws={'size': FS_ANNOT},
                    cbar_kws={'label': metric}, linewidths=0.5, ax=ax)

    ax.set_title(f'{task_name} Performance: {metric} ({hpo_mode})\nDatasets x Methods',
                 fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax.set_xticklabels(ax.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()

    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_heatmap_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal (transposed) ---
    n_h = max(pivot.shape[0], pivot.shape[1])
    fig_h, ax_h = plt.subplots(figsize=(max(12, n_h * 0.5), max(8, n_h * 0.4)))
    pivot_T = pivot.T
    if metric == 'R2':
        sns.heatmap(pivot_T, annot=True, fmt='.3f', cmap='RdYlGn',
                    center=0, vmin=-abs_max, vmax=abs_max, annot_kws={'size': FS_ANNOT},
                    cbar_kws={'label': metric}, linewidths=0.5, ax=ax_h)
    else:
        sns.heatmap(pivot_T, annot=True, fmt='.3f', cmap=cmap,
                    vmin=vmin, vmax=vmax, annot_kws={'size': FS_ANNOT},
                    cbar_kws={'label': metric}, linewidths=0.5, ax=ax_h)
    ax_h.set_title(f'{task_name} Performance: {metric} ({hpo_mode})\nMethods x Datasets',
                   fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax_h.set_xlabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax_h.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax_h.set_xticklabels(ax_h.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_ylabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_heatmap_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig, pivot


def create_rank_heatmap(df, metric, hpo_mode, task_name, figsize=(24, 12)):
    """
    Create a heatmap showing method ranks across datasets (1=best, n=worst).
    """
    mean_col = f'{metric}_mean'

    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found in data")
        return None, None, None, None

    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()

    if df_filtered.empty:
        print(f"Warning: No data for {hpo_mode}")
        return None, None, None, None

    pivot = df_filtered.pivot(index='dataset', columns='method', values=mean_col)
    rank_pivot = pivot.rank(axis=1, ascending=False, method='average')
    rank_pivot = rank_pivot.sort_index()
    method_avg_ranks = rank_pivot.mean(axis=0).sort_values()
    median_ranks = rank_pivot.median(axis=0).sort_values()
    rank_pivot = rank_pivot[method_avg_ranks.index]

    # --- Vertical ---
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(rank_pivot, annot=True, fmt='.1f', cmap='RdYlGn_r', annot_kws={'size': FS_ANNOT},
                cbar_kws={'label': 'Rank (1=best)'}, linewidths=0.5, ax=ax)
    ax.set_title(f'{task_name} Ranks: {metric} ({hpo_mode})\nDatasets x Methods (1=best)',
                 fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax.set_xticklabels(ax.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_ranks_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal ---
    n_h = max(pivot.shape[0], pivot.shape[1])
    fig_h, ax_h = plt.subplots(figsize=(max(12, n_h * 0.5), max(8, n_h * 0.4)))
    rank_T = rank_pivot.T
    sns.heatmap(rank_T, annot=True, fmt='.1f', cmap='RdYlGn_r', annot_kws={'size': FS_ANNOT},
                cbar_kws={'label': 'Rank (1=best)'}, linewidths=0.5, ax=ax_h)
    ax_h.set_title(f'{task_name} Ranks: {metric} ({hpo_mode})\nMethods x Datasets (1=best)',
                   fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax_h.set_xlabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax_h.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax_h.set_xticklabels(ax_h.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_ylabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_ranks_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig, rank_pivot, method_avg_ranks, median_ranks


def create_hpo_improvement_heatmap(df, metric, task_name, figsize=(24, 12)):
    """
    Create a heatmap showing improvement from NO_HPO to HPO.
    """
    mean_col = f'{metric}_mean'

    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found in data")
        return None, None

    no_hpo = df[df['hpo_mode'] == 'NO_HPO'].pivot(index='dataset', columns='method', values=mean_col)
    hpo = df[df['hpo_mode'] == 'HPO'].pivot(index='dataset', columns='method', values=mean_col)

    diff = hpo - no_hpo
    diff = diff.sort_index()
    method_avg_improvement = diff.mean(axis=0).sort_values(ascending=False)
    diff = diff[method_avg_improvement.index]

    fig, ax = plt.subplots(figsize=figsize)
    vmax = max(abs(diff.min().min()), abs(diff.max().max()))
    sns.heatmap(diff, annot=True, fmt='+.3f', cmap='RdYlGn', annot_kws={'size': FS_ANNOT},
                center=0, vmin=-vmax, vmax=vmax,
                cbar_kws={'label': f'{metric} Improvement (HPO - NO_HPO)'},
                linewidths=0.5, ax=ax)
    ax.set_title(f'{task_name}: HPO Impact on {metric}\nDatasets x Methods (green=improvement, red=degradation)',
                 fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax.set_xlabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax.set_xticklabels(ax.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()

    filename = FIGURES_DIR / f"{task_name.lower()}_hpo_improvement_{metric.lower()}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal ---
    n_h = max(diff.shape[0], diff.shape[1])
    fig_h, ax_h = plt.subplots(figsize=(max(12, n_h * 0.5), max(8, n_h * 0.4)))
    diff_T = diff.T
    sns.heatmap(diff_T, annot=True, fmt='+.3f', cmap='RdYlGn', annot_kws={'size': FS_ANNOT},
                center=0, vmin=-vmax, vmax=vmax,
                cbar_kws={'label': f'{metric} Improvement (HPO - NO_HPO)'},
                linewidths=0.5, ax=ax_h)
    ax_h.set_title(f'{task_name}: HPO Impact on {metric}\nMethods x Datasets (green=improvement, red=degradation)',
                   fontsize=FS_TITLE, fontweight='bold', pad=20)
    ax_h.set_xlabel('Dataset', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.tick_params(axis='x', labelsize=FS_TICK, rotation=45)
    ax_h.tick_params(axis='y', labelsize=FS_TICK_Y)
    ax_h.set_xticklabels(ax_h.get_xticklabels(), ha='right', rotation_mode='anchor')
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_ylabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_hpo_improvement_{metric.lower()}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    n_improvements = (diff > 0).sum().sum()
    n_degradations = (diff < 0).sum().sum()
    n_total = diff.notna().sum().sum()

    print(f"\nHPO Impact Statistics:")
    print(f"   Improvements: {n_improvements}/{n_total} ({100*n_improvements/n_total:.1f}%)")
    print(f"   Degradations: {n_degradations}/{n_total} ({100*n_degradations/n_total:.1f}%)")
    print(f"   Avg improvement: {diff.mean().mean():+.4f}")
    print(f"   Max improvement: {diff.max().max():+.4f}")
    print(f"   Max degradation: {diff.min().min():+.4f}")

    return fig, diff


def plot_average_performance_bar(df, metric, hpo_mode, task_name, figsize=(20, 8)):
    """
    Bar chart showing average performance across datasets with error bars.
    Generates both vertical and horizontal orientations.
    """
    mean_col = f'{metric}_mean'

    if mean_col not in df.columns:
        print(f"Warning: Metric '{metric}' not found in data")
        return None

    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()

    if df_filtered.empty:
        print(f"Warning: No data for {hpo_mode}")
        return None

    method_avg = df_filtered.groupby('method')[mean_col].mean().sort_values(ascending=False)
    method_std = df_filtered.groupby('method')[mean_col].std()

    # --- Vertical bar chart ---
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(method_avg))
    bars = ax.bar(x, method_avg.values, yerr=method_std.loc[method_avg.index].values,
                   capsize=5, alpha=0.8, edgecolor='black', linewidth=1.2)
    norm = plt.Normalize(vmin=method_avg.min(), vmax=method_avg.max())
    colors = plt.cm.RdYlGn(norm(method_avg.values))
    for bar, color in zip(bars, colors):
        bar.set_facecolor(color)
    ax.set_xticks(x)
    ax.set_xticklabels(method_avg.index, rotation=45, ha='right', fontsize=FS_TICK)
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel(f'{metric} (Mean +/- Std)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_title(f'{task_name}: Average {metric} Across Datasets ({hpo_mode})',
                 fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    for i, (val, std) in enumerate(zip(method_avg.values, method_std.loc[method_avg.index].values)):
        ax.text(i, val + std + 0.01, f'{val:.3f}', ha='center', va='bottom', fontsize=FS_BAR_LABEL)
    plt.tight_layout()
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_bar_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal bar chart ---
    n_h = len(method_avg)
    fig_h, ax_h = plt.subplots(figsize=(figsize[0] * 0.65, max(6, n_h * 0.4)))
    method_avg_asc = method_avg.sort_values(ascending=True)
    y = np.arange(len(method_avg_asc))
    bars_h = ax_h.barh(y, method_avg_asc.values,
                        xerr=method_std.loc[method_avg_asc.index].values,
                        capsize=5, alpha=0.8, edgecolor='black', linewidth=1.2)
    norm_h = plt.Normalize(vmin=method_avg_asc.min(), vmax=method_avg_asc.max())
    colors_h = plt.cm.RdYlGn(norm_h(method_avg_asc.values))
    for bar, color in zip(bars_h, colors_h):
        bar.set_facecolor(color)
    ax_h.set_yticks(y)
    ax_h.set_yticklabels(method_avg_asc.index, fontsize=FS_TICK_Y)
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_xlabel(f'{metric} (Mean +/- Std)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_title(f'{task_name}: Average {metric} Across Datasets ({hpo_mode})',
                   fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax_h.grid(axis='x', alpha=0.3)
    for i, (val, std) in enumerate(zip(method_avg_asc.values, method_std.loc[method_avg_asc.index].values)):
        ax_h.text(val + std + 0.005, i, f'{val:.3f}', ha='left', va='center', fontsize=FS_BAR_LABEL)
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_bar_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig


def plot_performance_distribution(df, metric, hpo_mode, task_name, figsize=(20, 8)):
    """
    Boxplot showing performance distribution across all folds.
    Generates both vertical and horizontal orientations.
    """
    if metric not in df.columns:
        print(f"Warning: Metric '{metric}' not found in raw data")
        return None

    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()

    if df_filtered.empty:
        print(f"Warning: No data for {hpo_mode}")
        return None

    method_order = df_filtered.groupby('method')[metric].median().sort_values(ascending=False).index

    # --- Vertical ---
    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(data=df_filtered, x='method', y=metric, order=method_order, palette='Set2', ax=ax)
    sns.stripplot(data=df_filtered, x='method', y=metric, order=method_order,
                  color='black', alpha=0.3, size=4, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=FS_TICK)
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel(metric, fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_xlabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_title(f'{task_name}: {metric} Distribution Across All Folds ({hpo_mode})',
                 fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_boxplot_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal ---
    method_order_h = df_filtered.groupby('method')[metric].median().sort_values(ascending=True).index
    n_h = len(method_order)
    fig_h, ax_h = plt.subplots(figsize=(figsize[0] * 0.65, max(6, n_h * 0.4)))
    sns.boxplot(data=df_filtered, y='method', x=metric, order=method_order_h, palette='Set2', ax=ax_h, orient='h')
    sns.stripplot(data=df_filtered, y='method', x=metric, order=method_order_h,
                  color='black', alpha=0.3, size=4, ax=ax_h, orient='h')
    ax_h.tick_params(axis='y', labelsize=FS_TICK_Y)
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_xlabel(metric, fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_ylabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_title(f'{task_name}: {metric} Distribution Across All Folds ({hpo_mode})',
                   fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax_h.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_boxplot_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig


def plot_rank_bar(avg_ranks, median_ranks, task_name, metric, hpo_mode, figsize=(20, 8)):
    """
    Bar chart showing average ranks. Generates both vertical and horizontal orientations.
    """
    # --- Vertical ---
    fig, ax = plt.subplots(figsize=figsize)
    x = np.arange(len(avg_ranks))
    bars = ax.bar(x, avg_ranks.values, alpha=0.8, edgecolor='black', linewidth=1.2)
    norm = plt.Normalize(vmin=avg_ranks.min(), vmax=avg_ranks.max())
    colors = plt.cm.RdYlGn_r(norm(avg_ranks.values))
    for bar, color in zip(bars, colors):
        bar.set_facecolor(color)
    ax.set_xticks(x)
    ax.set_xticklabels(avg_ranks.index, rotation=45, ha='right', fontsize=FS_TICK)
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel('Average Rank (lower is better)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_title(f'{task_name}: Average Ranks for {metric} ({hpo_mode})\nLabels: avg (median)',
                 fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    for i, val in enumerate(avg_ranks.values):
        med_val = median_ranks[avg_ranks.index[i]]
        ax.text(i, val + 0.3, f'{val:.1f} ({med_val:.1f})',
                ha='center', va='bottom', fontsize=FS_BAR_LABEL, fontweight='bold')
    plt.tight_layout()
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_bar_rank_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal ---
    avg_ranks_asc = avg_ranks.sort_values(ascending=False)
    n_h = len(avg_ranks_asc)
    fig_h, ax_h = plt.subplots(figsize=(figsize[0] * 0.65, max(6, n_h * 0.4)))
    y = np.arange(len(avg_ranks_asc))
    bars_h = ax_h.barh(y, avg_ranks_asc.values, alpha=0.8, edgecolor='black', linewidth=1.2)
    norm_h = plt.Normalize(vmin=avg_ranks_asc.min(), vmax=avg_ranks_asc.max())
    colors_h = plt.cm.RdYlGn_r(norm_h(avg_ranks_asc.values))
    for bar, color in zip(bars_h, colors_h):
        bar.set_facecolor(color)
    ax_h.set_yticks(y)
    ax_h.set_yticklabels(avg_ranks_asc.index, fontsize=FS_TICK_Y)
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_xlabel('Average Rank (lower is better)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_title(f'{task_name}: Average Ranks for {metric} ({hpo_mode})',
                   fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax_h.grid(axis='x', alpha=0.3)
    for i, val in enumerate(avg_ranks_asc.values):
        med_val = median_ranks[avg_ranks_asc.index[i]]
        ax_h.text(val + 0.3, i, f'{val:.1f} ({med_val:.1f})',
                  ha='left', va='center', fontsize=FS_BAR_LABEL)
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_bar_rank_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig


def plot_rank_distribution(df, metric, hpo_mode, task_name, figsize=(20, 8)):
    """
    Boxplot showing rank distribution across all folds.
    Generates both vertical and horizontal orientations.
    """
    if metric not in df.columns:
        print(f"Warning: Metric '{metric}' not found in raw data")
        return None

    df_filtered = df[df['hpo_mode'] == hpo_mode].copy()

    if df_filtered.empty:
        print(f"Warning: No data for {hpo_mode}")
        return None

    df_filtered['rank'] = df_filtered.groupby(['dataset', 'fold_id'])[metric].rank(
        ascending=False, method='average'
    )
    method_order = df_filtered.groupby('method')['rank'].median().sort_values(ascending=True).index

    # --- Vertical ---
    fig, ax = plt.subplots(figsize=figsize)
    sns.boxplot(data=df_filtered, x='method', y='rank', order=method_order, palette='Set2', ax=ax)
    sns.stripplot(data=df_filtered, x='method', y='rank', order=method_order,
                  color='black', alpha=0.3, size=4, ax=ax)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=FS_TICK)
    for label in ax.get_xticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_ylabel(f'{metric} Rank (1 = best)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_xlabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax.set_title(f'{task_name}: {metric} Rank Distribution Across All Folds ({hpo_mode})',
                 fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax.grid(axis='y', alpha=0.3)
    ax.invert_yaxis()
    plt.tight_layout()
    hpo_suffix = hpo_mode.lower()
    filename = FIGURES_DIR / f"{task_name.lower()}_boxplot_rank_{metric.lower()}_{hpo_suffix}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename.name}")

    # --- Horizontal ---
    method_order_h = df_filtered.groupby('method')['rank'].median().sort_values(ascending=False).index
    n_h = len(method_order_h)
    fig_h, ax_h = plt.subplots(figsize=(figsize[0] * 0.65, max(6, n_h * 0.4)))
    sns.boxplot(data=df_filtered, y='method', x='rank', order=method_order_h, palette='Set2', ax=ax_h, orient='h')
    sns.stripplot(data=df_filtered, y='method', x='rank', order=method_order_h,
                  color='black', alpha=0.3, size=4, ax=ax_h, orient='h')
    ax_h.tick_params(axis='y', labelsize=FS_TICK_Y)
    for label in ax_h.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax_h.set_xlabel(f'{metric} Rank (1 = best)', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_ylabel('Method', fontsize=FS_AXIS_LABEL, fontweight='bold')
    ax_h.set_title(f'{task_name}: {metric} Rank Distribution Across All Folds ({hpo_mode})',
                   fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax_h.grid(axis='x', alpha=0.3)
    ax_h.invert_xaxis()
    plt.tight_layout()
    filename_h = FIGURES_DIR / f"{task_name.lower()}_boxplot_rank_{metric.lower()}_{hpo_suffix}_horizontal.png"
    plt.savefig(filename_h, dpi=300, bbox_inches='tight')
    print(f"Saved: {filename_h.name}")

    return fig

---
# 📊 Part A: PD Analysis (Classification)

**Primary Metric**: AUC (Area Under ROC Curve)  
Range: 0.5-1.0, where 0.5 = random, 1.0 = perfect

## A1. Performance Heatmaps

### A1.1 NO_HPO Performance Matrix

In [ ]:
print("Creating PD AUC heatmap (NO_HPO)...")
fig_pd_no_hpo, pivot_pd_no_hpo = create_performance_heatmap(
    pd_agg, 'AUC', 'NO_HPO', 'PD', cmap='RdYlGn'
)
plt.show()

### A1.2 HPO Performance Matrix

In [ ]:
print("Creating PD AUC heatmap (HPO)...")
fig_pd_hpo, pivot_pd_hpo = create_performance_heatmap(
    pd_agg, 'AUC', 'HPO', 'PD', cmap='RdYlGn'
)
plt.show()

### A1.3 HPO Improvement Matrix

In [ ]:
print("Creating PD HPO improvement heatmap...")
fig_pd_improvement, diff_pd = create_hpo_improvement_heatmap(
    pd_agg, 'AUC', 'PD'
)
plt.show()

## A2. Performance Distributions

### A2.1 Average Performance Bar Charts

In [ ]:
print("Creating PD average performance bar chart (NO_HPO)...")
plot_average_performance_bar(pd_agg, 'AUC', 'NO_HPO', 'PD')
plt.show()

print("\nCreating PD average performance bar chart (HPO)...")
plot_average_performance_bar(pd_agg, 'AUC', 'HPO', 'PD')
plt.show()

### A2.2 Performance Distribution Boxplots

In [ ]:
print("Creating PD AUC distribution boxplot (NO_HPO)...")
plot_performance_distribution(pd_raw, 'AUC', 'NO_HPO', 'PD')
plt.show()

print("\nCreating PD AUC distribution boxplot (HPO)...")
plot_performance_distribution(pd_raw, 'AUC', 'HPO', 'PD')
plt.show()

### A2.3 Rank Distribution Boxplots

In [ ]:
print("Creating PD AUC rank distribution boxplot (NO_HPO)...")
plot_rank_distribution(pd_raw, 'AUC', 'NO_HPO', 'PD')
plt.show()

print("\nCreating PD AUC rank distribution boxplot (HPO)...")
plot_rank_distribution(pd_raw, 'AUC', 'HPO', 'PD')
plt.show()

## A3. Rank Analysis

### A3.1 Rank Heatmaps

In [ ]:
print("Creating PD rank heatmap (NO_HPO)...")
fig_pd_ranks_no_hpo, ranks_pd_no_hpo, avg_ranks_pd_no_hpo, median_ranks_pd_no_hpo = create_rank_heatmap(
    pd_agg, 'AUC', 'NO_HPO', 'PD'
)
plt.show()

print("\nMethod Rankings (NO_HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_pd_no_hpo.items(), 1):
    med_rank = median_ranks_pd_no_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

In [ ]:
print("Creating PD rank heatmap (HPO)...")
fig_pd_ranks_hpo, ranks_pd_hpo, avg_ranks_pd_hpo, median_ranks_pd_hpo = create_rank_heatmap(
    pd_agg, 'AUC', 'HPO', 'PD'
)
plt.show()

print("\nMethod Rankings (HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_pd_hpo.items(), 1):
    med_rank = median_ranks_pd_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

### A3.2 Average Rank Bar Charts

In [ ]:
if avg_ranks_pd_no_hpo is not None:
    print("Creating PD average rank bar chart (NO_HPO)...")
    plot_rank_bar(avg_ranks_pd_no_hpo, median_ranks_pd_no_hpo, 'PD', 'AUC', 'NO_HPO')
    plt.show()

if avg_ranks_pd_hpo is not None:
    print("\nCreating PD average rank bar chart (HPO)...")
    plot_rank_bar(avg_ranks_pd_hpo, median_ranks_pd_hpo, 'PD', 'AUC', 'HPO')
    plt.show()

## A4. Statistical Testing

### A4.1 PAMA Analysis

In [ ]:
print("\n" + "=" * 80)
print("  PD - PAMA ANALYSIS (HPO) - Fold-Level")
print("=" * 80)

# Use raw data to calculate PAMA at fold level
pd_hpo_raw = pd_raw[pd_raw['hpo_mode'] == 'HPO'].copy()

if not pd_hpo_raw.empty and 'AUC' in pd_hpo_raw.columns:
    pama_scores = {}
    
    # Group by dataset and fold to find winner for each fold
    for (dataset, fold), group in pd_hpo_raw.groupby(['dataset', 'fold_id']):
        if len(group) > 0 and 'AUC' in group.columns:
            max_score = group['AUC'].max()
            best_methods = group[group['AUC'] >= max_score - 0.0001]['method'].tolist()
            
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Total number of folds
    n_folds = len(pd_hpo_raw.groupby(['dataset', 'fold_id']))
    
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama_pct': 100 * score / n_folds}
        for method, score in pama_scores.items()
    ]).sort_values('pama_pct', ascending=False)
    
    print(f"\nPAMA Scores (across {n_folds} folds):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama_pct'] / 2)
        print(f"  {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama_pct']:5.1f}%  {bar}")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.RdYlGn(pama_df['pama_pct'] / pama_df['pama_pct'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=colors, edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])

    # Color foundation model labels red
    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'PD: Probability of Achieving Maximum AUC (HPO)\nFold-Level Analysis (n={n_folds} folds)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama_pct'] + 1, i, f"{row['pama_pct']:.1f}%", va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = FIGURES_DIR / "pd_pama_analysis_hpo.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {filename.name}")
    plt.show()
else:
    print("⚠️  No HPO data available for PAMA analysis")

### A4.2 Full PAMA (All Methods)

In [ ]:
# Full PAMA including methods with 0% wins
print("\n" + "=" * 80)
print("  PD - FULL PAMA ANALYSIS (HPO) - All Methods Including 0%")
print("=" * 80)

pd_hpo_raw_full = pd_raw[pd_raw['hpo_mode'] == 'HPO'].copy()

if not pd_hpo_raw_full.empty and 'AUC' in pd_hpo_raw_full.columns:
    pama_scores = {}

    for (dataset, fold), group in pd_hpo_raw_full.groupby(['dataset', 'fold_id']):
        if len(group) > 0:
            max_score = group['AUC'].max()
            best_methods = group[group['AUC'] >= max_score - 0.0001]['method'].tolist()
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))

    n_folds = len(pd_hpo_raw_full.groupby(['dataset', 'fold_id']))
    all_methods = sorted(pd_hpo_raw_full['method'].unique())

    pama_df = pd.DataFrame([
        {'method': m, 'wins': pama_scores.get(m, 0), 'pama_pct': 100 * pama_scores.get(m, 0) / n_folds}
        for m in all_methods
    ]).sort_values('pama_pct', ascending=False)

    fig, ax = plt.subplots(figsize=(14, max(8, len(pama_df) * 0.45)))

    bar_colors = []
    for _, row in pama_df.iterrows():
        if row['method'] in FOUNDATION_METHODS:
            bar_colors.append('indianred')
        elif row['pama_pct'] > 0:
            bar_colors.append('steelblue')
        else:
            bar_colors.append('lightgray')

    ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=bar_colors, edgecolor='black', linewidth=0.8, alpha=0.85)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])

    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')

    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'PD: PAMA (AUC) - All Methods (HPO)\nn={n_folds} folds | Red bars = Foundation Models',
                 fontsize=13, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()

    for i, (idx, row) in enumerate(pama_df.iterrows()):
        label = f"{row['pama_pct']:.1f}%" if row['pama_pct'] > 0 else "0%"
        ax.text(max(row['pama_pct'] + 0.5, 1.5), i, label, va='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    filename = FIGURES_DIR / "pd_pama_full_all_methods.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n\u2705 Saved: {filename.name}")
    plt.show()


### A4.3 Foundation Model Spotlight

In [ ]:
# Foundation Model Spotlight: How do foundation models compare across dataset sizes?
print("\n" + "=" * 80)
print("  PD - FOUNDATION MODEL SPOTLIGHT")
print("=" * 80)

pd_hpo_raw_spot = pd_raw[pd_raw['hpo_mode'] == 'HPO'].copy()

if not pd_hpo_raw_spot.empty and 'AUC' in pd_hpo_raw_spot.columns:
    # --- 1. Average AUC: Foundation vs Non-Foundation ---
    avg_auc = pd_hpo_raw_spot.groupby('method')['AUC'].mean().reset_index()
    avg_auc['is_foundation'] = avg_auc['method'].isin(FOUNDATION_METHODS)
    avg_auc = avg_auc.sort_values('AUC', ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(avg_auc) * 0.4)))
    bar_colors = ['indianred' if f else 'steelblue' for f in avg_auc['is_foundation']]
    ax.barh(range(len(avg_auc)), avg_auc['AUC'], color=bar_colors, edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.set_yticks(range(len(avg_auc)))
    ax.set_yticklabels(avg_auc['method'])
    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')

    ax.set_xlabel('Average AUC (across all datasets & folds)', fontweight='bold')
    ax.set_title('PD: Average AUC per Method (HPO)\nRed = Foundation Models', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    for i, (_, row) in enumerate(avg_auc.iterrows()):
        ax.text(row['AUC'] + 0.001, i, f"{row['AUC']:.4f}", va='center', fontsize=8)

    plt.tight_layout()
    filename = FIGURES_DIR / "pd_foundation_spotlight_avg_auc.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\u2705 Saved: {filename.name}")
    plt.show()

    # --- 2. Win rate: Foundation model is best vs any non-foundation is best ---
    print("\nFoundation vs Non-Foundation Win Rate (per fold):")
    fm_wins = 0
    nfm_wins = 0
    ties = 0
    n_total = 0

    for (dataset, fold), group in pd_hpo_raw_spot.groupby(['dataset', 'fold_id']):
        n_total += 1
        fm_group = group[group['method'].isin(FOUNDATION_METHODS)]
        nfm_group = group[~group['method'].isin(FOUNDATION_METHODS)]

        if fm_group.empty or nfm_group.empty:
            continue

        best_fm = fm_group['AUC'].max()
        best_nfm = nfm_group['AUC'].max()

        if best_fm > best_nfm + 0.0001:
            fm_wins += 1
        elif best_nfm > best_fm + 0.0001:
            nfm_wins += 1
        else:
            ties += 1

    labels = ['Foundation\nModel Wins', 'Non-Foundation\nWins', 'Ties']
    counts = [fm_wins, nfm_wins, ties]
    colors_pie = ['indianred', 'steelblue', 'lightgray']

    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(counts, labels=labels, colors=colors_pie,
                                       autopct='%1.1f%%', startangle=90,
                                       textprops={'fontsize': 12, 'fontweight': 'bold'})
    for autotext in autotexts:
        autotext.set_fontsize(13)
        autotext.set_fontweight('bold')
    ax.set_title(f'PD: Foundation vs Non-Foundation Win Rate (HPO)\n{n_total} total folds',
                 fontsize=13, fontweight='bold', pad=20)

    plt.tight_layout()
    filename = FIGURES_DIR / "pd_foundation_vs_nonfoundation_winrate.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\u2705 Saved: {filename.name}")
    plt.show()

    print(f"\n  Foundation model best: {fm_wins}/{n_total} ({100*fm_wins/n_total:.1f}%)")
    print(f"  Non-foundation best:  {nfm_wins}/{n_total} ({100*nfm_wins/n_total:.1f}%)")
    print(f"  Ties:                 {ties}/{n_total} ({100*ties/n_total:.1f}%)")


### A4.4 Statistical Analysis Helper Functions

In [ ]:
# =============================================================================
# STATISTICAL ANALYSIS HELPER FUNCTIONS
# =============================================================================
from itertools import combinations
from scipy.stats import friedmanchisquare, wilcoxon, ttest_rel, rankdata
from statsmodels.stats.multitest import multipletests
import matplotlib.patches as mpatches

def compute_performance_matrix(raw_df, metric, hpo_mode='HPO'):
    """
    Build a (dataset x fold) x method matrix of performance scores.
    Each row is one (dataset, fold_id) observation; each column is a method.
    Only methods present in ALL (dataset, fold) combinations are retained.
    """
    df = raw_df[raw_df['hpo_mode'] == hpo_mode].copy()
    if metric not in df.columns:
        return None
    pivot = df.pivot_table(index=['dataset', 'fold_id'], columns='method', values=metric)
    # Drop methods with any missing observations
    pivot = pivot.dropna(axis=1)
    return pivot


def friedman_test(perf_matrix):
    """Run Friedman test on a performance matrix (observations x methods)."""
    k = perf_matrix.shape[1]
    if k < 3:
        return None, None
    stat, p = friedmanchisquare(*[perf_matrix.iloc[:, i] for i in range(k)])
    return stat, p


def pairwise_ttest_holm(perf_matrix):
    """
    Pairwise paired T-tests with Holm correction.
    Returns a DataFrame of (method_a, method_b, t_stat, p_raw, p_corrected, significant).
    """
    methods = list(perf_matrix.columns)
    results = []
    for ma, mb in combinations(methods, 2):
        t_stat, p_raw = ttest_rel(perf_matrix[ma], perf_matrix[mb])
        results.append({'method_a': ma, 'method_b': mb, 't_stat': t_stat, 'p_raw': p_raw})

    if not results:
        return pd.DataFrame()

    res_df = pd.DataFrame(results)
    # Holm correction
    reject, p_corrected, _, _ = multipletests(res_df['p_raw'], alpha=0.05, method='holm')
    res_df['p_corrected'] = p_corrected
    res_df['significant'] = reject
    return res_df


def compute_average_ranks(perf_matrix, higher_is_better=True):
    """Compute average ranks across observations. Rank 1 = best."""
    if higher_is_better:
        ranks = perf_matrix.rank(axis=1, ascending=False, method='average')
    else:
        ranks = perf_matrix.rank(axis=1, ascending=True, method='average')
    return ranks.mean().sort_values()


def plot_critical_difference_diagram(
    avg_ranks, pairwise_df, title, figsize=(16, 8), alpha=0.05,
    foundation_methods=None, save_path=None
):
    """
    Plot a Critical Difference (CD) diagram (Demsar, 2006 style).

    Methods are placed on a number line by their average rank.
    Horizontal bars connect methods whose pairwise difference is NOT
    statistically significant (p_corrected > alpha).
    """
    if foundation_methods is None:
        foundation_methods = set()

    methods = list(avg_ranks.index)
    ranks = avg_ranks.values
    n_methods = len(methods)

    fig, ax = plt.subplots(figsize=figsize)

    # Number line
    rank_min, rank_max = 1, n_methods
    margin = 0.5
    ax.set_xlim(rank_min - margin, rank_max + margin)
    ax.set_ylim(-1.5, n_methods + 1.5)

    # Draw the rank axis at the top
    ax.hlines(n_methods + 0.3, rank_min, rank_max, color='black', linewidth=1.5)
    for r in range(1, n_methods + 1):
        ax.vlines(r, n_methods + 0.1, n_methods + 0.5, color='black', linewidth=1)
        ax.text(r, n_methods + 0.65, str(r), ha='center', va='bottom', fontsize=FS_TICK)

    # Split methods into two halves: left (best) and right (worst)
    n_left = (n_methods + 1) // 2
    n_right = n_methods - n_left

    y_positions = {}

    # Left side (best ranks) - top to bottom
    for i in range(n_left):
        y_pos = n_methods - 1 - i * (n_methods / max(n_left, 1))
        y_positions[methods[i]] = y_pos
        rank_val = ranks[i]

        is_fm = methods[i] in foundation_methods
        color = 'red' if is_fm else 'black'
        weight = 'bold' if is_fm else 'normal'

        # Line from method name to rank position
        ax.hlines(y_pos, rank_min - margin + 0.1, rank_val, color='gray', linewidth=0.5, linestyle='-')
        ax.plot(rank_val, y_pos, 'o', color=color, markersize=5, zorder=5)
        ax.text(rank_min - margin + 0.05, y_pos, f'{methods[i]} ({rank_val:.2f})',
                ha='right', va='center', fontsize=FS_TICK_Y, color=color, fontweight=weight)

    # Right side (worst ranks) - top to bottom
    for j in range(n_right):
        i = n_left + j
        y_pos = n_methods - 1 - j * (n_methods / max(n_right, 1))
        y_positions[methods[i]] = y_pos
        rank_val = ranks[i]

        is_fm = methods[i] in foundation_methods
        color = 'red' if is_fm else 'black'
        weight = 'bold' if is_fm else 'normal'

        ax.hlines(y_pos, rank_val, rank_max + margin - 0.1, color='gray', linewidth=0.5, linestyle='-')
        ax.plot(rank_val, y_pos, 'o', color=color, markersize=5, zorder=5)
        ax.text(rank_max + margin - 0.05, y_pos, f'({rank_val:.2f}) {methods[i]}',
                ha='left', va='center', fontsize=FS_TICK_Y, color=color, fontweight=weight)

    # Draw cliques (non-significant groups) as thick horizontal bars
    if not pairwise_df.empty:
        nonsig = pairwise_df[~pairwise_df['significant']]
        rank_lookup = dict(zip(methods, ranks))

        bars = []
        for _, row in nonsig.iterrows():
            r_a = rank_lookup.get(row['method_a'])
            r_b = rank_lookup.get(row['method_b'])
            if r_a is not None and r_b is not None:
                bars.append((min(r_a, r_b), max(r_a, r_b)))

        # Merge overlapping bars into cliques
        if bars:
            bars.sort()
            merged = [bars[0]]
            for start, end in bars[1:]:
                if start <= merged[-1][1]:
                    merged[-1] = (merged[-1][0], max(merged[-1][1], end))
                else:
                    merged.append((start, end))

            # Draw bars at different y-levels to avoid overlap
            bar_y = -0.3
            for start, end in merged:
                ax.hlines(bar_y, start - 0.05, end + 0.05, color='black', linewidth=3.5)
                bar_y -= 0.35

    ax.set_title(title, fontsize=FS_TITLE, fontweight='bold', pad=30)
    ax.axis('off')
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {save_path.name}")
    plt.show()


def plot_pairwise_significance_heatmap(pairwise_df, avg_ranks, title, save_path=None):
    """
    Plot a heatmap of pairwise p-values (Holm-corrected).
    Methods ordered by average rank.
    """
    methods = list(avg_ranks.index)
    n = len(methods)

    # Build symmetric p-value matrix
    p_matrix = pd.DataFrame(np.nan, index=methods, columns=methods)
    for _, row in pairwise_df.iterrows():
        if row['method_a'] in methods and row['method_b'] in methods:
            p_matrix.loc[row['method_a'], row['method_b']] = row['p_corrected']
            p_matrix.loc[row['method_b'], row['method_a']] = row['p_corrected']

    # Fill diagonal with 1.0
    np.fill_diagonal(p_matrix.values, 1.0)

    fig, ax = plt.subplots(figsize=(max(14, n * 0.6), max(12, n * 0.5)))

    # Create annotation: show p-value and mark significance
    annot = p_matrix.copy().astype(str)
    for i in range(n):
        for j in range(n):
            val = p_matrix.iloc[i, j]
            if i == j:
                annot.iloc[i, j] = '-'
            elif pd.isna(val):
                annot.iloc[i, j] = ''
            elif val < 0.001:
                annot.iloc[i, j] = '<.001***'
            elif val < 0.01:
                annot.iloc[i, j] = f'{val:.3f}**'
            elif val < 0.05:
                annot.iloc[i, j] = f'{val:.3f}*'
            else:
                annot.iloc[i, j] = f'{val:.3f}'

    sns.heatmap(
        p_matrix.astype(float), ax=ax, annot=annot, fmt='',
        cmap='RdYlGn', vmin=0, vmax=0.1, center=0.05,
        linewidths=0.5, linecolor='white',
        annot_kws={'fontsize': FS_ANNOT},
        cbar_kws={'label': 'Holm-corrected p-value'}
    )

    ax.set_title(title, fontsize=FS_TITLE, fontweight='bold', pad=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right', fontsize=FS_TICK)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=FS_TICK)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {save_path.name}")
    plt.show()

print("Statistical analysis helper functions loaded.")


# ============================================================================
# PAMA (Probability of Achieving Maximum Accuracy) Analysis
# ============================================================================

# Method family mapping for combined analysis (paper Section 5.3):
# "We treat SVM and SVR, as well as linear and logistic regression,
#  as members of the same model family."
METHOD_FAMILY_MAP = {
    'svm': 'SVM/SVR',
    'svr': 'SVM/SVR',
    'linear_regression': 'Linear/Logistic',
    'logistic_regression': 'Linear/Logistic',
}


def get_method_family(method_name):
    """Map a method to its family name (for combined PD+LGD analysis)."""
    return METHOD_FAMILY_MAP.get(method_name, method_name)


def compute_pama(perf_matrix, higher_is_better=True):
    """
    Compute PAMA: Probability of Achieving Maximum Accuracy (Delgado et al., 2014).

    For each observation (dataset x fold), determines which method achieved the
    best performance. Returns a Series of win fractions per method.

    Parameters
    ----------
    perf_matrix : pd.DataFrame
        (observations x methods) matrix of performance scores.
    higher_is_better : bool
        If True, maximum value is best; if False, minimum is best.

    Returns
    -------
    pd.Series : fraction of observations where each method achieved the best score.
    """
    if higher_is_better:
        best_per_row = perf_matrix.idxmax(axis=1)
    else:
        best_per_row = perf_matrix.idxmin(axis=1)
    win_counts = best_per_row.value_counts()
    # Include methods with zero wins
    for m in perf_matrix.columns:
        if m not in win_counts.index:
            win_counts[m] = 0
    win_fractions = (win_counts / len(perf_matrix)).sort_values(ascending=False)
    return win_fractions


def compute_pama_by_family(perf_matrix, higher_is_better=True):
    """
    PAMA analysis with method family grouping.

    For methods mapped to the same family (e.g., SVM+SVR), the family
    'wins' if ANY member achieves the best score in a fold.
    """
    # For each row, find the best score
    if higher_is_better:
        best_vals = perf_matrix.max(axis=1)
    else:
        best_vals = perf_matrix.min(axis=1)

    # Map each method to its family
    families = {m: get_method_family(m) for m in perf_matrix.columns}
    unique_families = sorted(set(families.values()))

    family_wins = {f: 0 for f in unique_families}
    for idx in perf_matrix.index:
        row = perf_matrix.loc[idx]
        best_val = best_vals[idx]
        # Find all methods that achieved the best
        winners = row[row == best_val].index.tolist()
        # Map to families (each family counts once)
        winning_families = set(families[w] for w in winners)
        for fam in winning_families:
            family_wins[fam] += 1

    total = len(perf_matrix)
    family_fractions = pd.Series({f: family_wins[f] / total for f in unique_families})
    return family_fractions.sort_values(ascending=False)


def plot_pama(win_fractions, title, foundation_methods=None, figsize=(14, 8),
              save_path=None, top_n=None):
    """
    Plot PAMA results as a horizontal bar chart.

    Parameters
    ----------
    win_fractions : pd.Series
        Method -> fraction of wins, sorted descending.
    title : str
    foundation_methods : set, optional
    figsize : tuple
    save_path : Path, optional
    top_n : int, optional
        Only show top N methods (those with > 0 wins shown by default).
    """
    if foundation_methods is None:
        foundation_methods = set()

    # Filter to methods with > 0 wins, or top_n
    if top_n:
        data = win_fractions.head(top_n)
    else:
        data = win_fractions[win_fractions > 0]
        if len(data) == 0:
            data = win_fractions.head(5)

    # Sort ascending for horizontal bar (bottom = best)
    data = data.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=figsize)
    colors = ['#e74c3c' if m in foundation_methods else '#3498db' for m in data.index]
    bars = ax.barh(range(len(data)), data.values * 100, color=colors, edgecolor='white', height=0.7)

    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index, fontsize=FS_TICK_Y)
    ax.set_xlabel('Win Rate (%)', fontsize=FS_AXIS_LABEL)
    ax.set_title(title, fontsize=FS_SUBTITLE, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3)

    # Add percentage labels
    for bar, val in zip(bars, data.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                f'{val*100:.1f}%', ha='left', va='center', fontsize=FS_BAR_LABEL, fontweight='bold')

    ax.set_xlim(0, max(data.values) * 100 * 1.15)

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='#e74c3c', label='Foundation Model'),
                       Patch(facecolor='#3498db', label='Other')]
    ax.legend(handles=legend_elements, loc='lower right', fontsize=FS_LEGEND)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        print(f"  Saved: {save_path.name}")
    plt.show()


def group_performance_by_family(perf_matrix):
    """
    Group methods into families by taking the best performance per family per observation.

    For combined PD+LGD analysis, methods like SVM and SVR are treated as one family.
    For each (dataset, fold), the family score is the best score among its members.
    """
    families = {}
    for method in perf_matrix.columns:
        family = get_method_family(method)
        if family not in families:
            families[family] = []
        families[family].append(method)

    family_perf = pd.DataFrame(index=perf_matrix.index)
    for family, members in sorted(families.items()):
        family_perf[family] = perf_matrix[members].max(axis=1)

    return family_perf

### A4.5 Pairwise Statistical Comparisons & Critical Difference Diagram

Friedman omnibus test, pairwise paired T-tests with Holm correction, and Critical Difference diagram (Demsar, 2006) for PD methods.

In [ ]:
print("\n" + "=" * 80)
print("  PD - PAIRWISE STATISTICAL COMPARISONS (HPO)")
print("=" * 80)

# Build performance matrix: (dataset x fold) x method
pd_perf = compute_performance_matrix(pd_raw, 'AUC', hpo_mode='HPO')

if pd_perf is not None and pd_perf.shape[1] >= 3:
    print(f"\nPerformance matrix: {pd_perf.shape[0]} observations x {pd_perf.shape[1]} methods")

    # --- Friedman Test ---
    f_stat, f_p = friedman_test(pd_perf)
    print("\n" + "\u2500" * 60)
    print(f"Friedman test:  \u03c7\u00b2 = {f_stat:.2f},  p = {f_p:.2e}")
    if f_p < 0.05:
        print("  -> Significant differences exist among methods (p < 0.05)")
    else:
        print("  -> No significant differences detected (p >= 0.05)")
    print("\u2500" * 60)

    # --- Average Ranks ---
    pd_avg_ranks = compute_average_ranks(pd_perf, higher_is_better=True)
    print(f"\nAverage Ranks (1 = best):")
    for method, rank in pd_avg_ranks.items():
        marker = " *" if method in FOUNDATION_METHODS else ""
        print(f"  {method:20s}  {rank:.2f}{marker}")

    # --- Pairwise T-tests with Holm correction ---
    pd_pairwise = pairwise_ttest_holm(pd_perf)
    n_sig = pd_pairwise['significant'].sum()
    n_total = len(pd_pairwise)
    print(f"\nPairwise T-tests (Holm correction):")
    print(f"  {n_sig}/{n_total} pairs significantly different (alpha=0.05)")

    # Show top significant differences
    sig_pairs = pd_pairwise[pd_pairwise['significant']].sort_values('p_corrected')
    if len(sig_pairs) > 0:
        print(f"\n  Top significant pairs:")
        for _, row in sig_pairs.head(15).iterrows():
            print(f"    {row['method_a']:18s} vs {row['method_b']:18s}  p={row['p_corrected']:.4f}")

    # --- Critical Difference Diagram ---
    print("\n" + "\u2500" * 60)
    print("Generating Critical Difference diagram...")
    plot_critical_difference_diagram(
        pd_avg_ranks, pd_pairwise,
        title=f'PD: Critical Difference Diagram (AUC, HPO)\nFriedman p={f_p:.2e}  |  {pd_perf.shape[0]} observations',
        figsize=(18, max(10, len(pd_avg_ranks) * 0.45)),
        foundation_methods=FOUNDATION_METHODS,
        save_path=FIGURES_DIR / "pd_critical_difference_auc_hpo.png"
    )

    # --- Pairwise Significance Heatmap ---
    print("Generating pairwise significance heatmap...")
    plot_pairwise_significance_heatmap(
        pd_pairwise, pd_avg_ranks,
        title='PD: Pairwise T-Test p-values (Holm-corrected, AUC HPO)',
        save_path=FIGURES_DIR / "pd_pairwise_significance_heatmap_auc_hpo.png"
    )

    # Horizontal variant
    plot_critical_difference_diagram(
        pd_avg_ranks, pd_pairwise,
        title=f'PD: Critical Difference Diagram (AUC, HPO)\nFriedman p={f_p:.2e}',
        figsize=(20, max(10, len(pd_avg_ranks) * 0.45)),
        foundation_methods=FOUNDATION_METHODS,
        save_path=FIGURES_DIR / "pd_critical_difference_auc_hpo_horizontal.png"
    )
else:
    print("Warning: Insufficient data for PD statistical analysis")

### A4.6 PAMA Analysis (Probability of Achieving Maximum Accuracy)

Following Delgado et al. (2014), we compute the fraction of (dataset, fold) observations
where each method achieves the best performance score.

In [ ]:
print("\n" + "=" * 80)
print("  PD - PAMA ANALYSIS (Probability of Achieving Maximum Accuracy)")
print("=" * 80)

for hpo_mode in ['HPO', 'NO_HPO']:
    pd_perf_pama = compute_performance_matrix(pd_raw, 'AUC', hpo_mode=hpo_mode)
    if pd_perf_pama is not None:
        pama = compute_pama(pd_perf_pama, higher_is_better=True)
        n_obs = len(pd_perf_pama)
        print(f"\nPAMA - PD AUC ({hpo_mode}): {n_obs} observations, {len(pd_perf_pama.columns)} methods")
        print("-" * 60)
        for method, frac in pama.items():
            marker = " *" if method in FOUNDATION_METHODS else ""
            print(f"  {method:25s}  {frac*100:5.1f}%  ({int(frac*n_obs):3d}/{n_obs} wins){marker}")

        # Foundation models as a group
        fm_wins = sum(frac for m, frac in pama.items() if m in FOUNDATION_METHODS)
        print(f"\n  Foundation models (group): {fm_wins*100:.1f}%")

        plot_pama(
            pama,
            title=f'PD: PAMA Analysis (AUC, {hpo_mode})\n{n_obs} observations',
            foundation_methods=FOUNDATION_METHODS,
            figsize=(14, max(6, sum(1 for v in pama.values if v > 0) * 0.45)),
            save_path=FIGURES_DIR / f"pd_pama_auc_{hpo_mode.lower()}.png"
        )

---
# 📈 Part B: LGD Analysis (Regression)

**Primary Metric**: R² (Coefficient of Determination)  
Range: -∞ to 1.0, where 0 = baseline, 1.0 = perfect, <0 = worse than baseline

## B1. Performance Heatmaps

### B1.1 NO_HPO Performance Matrix

In [ ]:
print("Creating LGD R² heatmap (NO_HPO)...")
fig_lgd_no_hpo, pivot_lgd_no_hpo = create_performance_heatmap(
    lgd_agg, 'R2', 'NO_HPO', 'LGD', cmap='RdYlGn'
)
plt.show()

### B1.2 HPO Performance Matrix

In [ ]:
print("Creating LGD R² heatmap (HPO)...")
fig_lgd_hpo, pivot_lgd_hpo = create_performance_heatmap(
    lgd_agg, 'R2', 'HPO', 'LGD', cmap='RdYlGn'
)
plt.show()

### B1.3 HPO Improvement Matrix

In [ ]:
print("Creating LGD HPO improvement heatmap...")
fig_lgd_improvement, diff_lgd = create_hpo_improvement_heatmap(
    lgd_agg, 'R2', 'LGD'
)
plt.show()

## B2. Performance Distributions

### B2.1 Average Performance Bar Charts

In [ ]:
print("Creating LGD average performance bar chart (NO_HPO)...")
plot_average_performance_bar(lgd_agg, 'R2', 'NO_HPO', 'LGD')
plt.show()

print("\nCreating LGD average performance bar chart (HPO)...")
plot_average_performance_bar(lgd_agg, 'R2', 'HPO', 'LGD')
plt.show()

### B2.2 Performance Distribution Boxplots

In [ ]:
print("Creating LGD R² distribution boxplot (NO_HPO)...")
plot_performance_distribution(lgd_raw, 'R2', 'NO_HPO', 'LGD')
plt.show()

print("\nCreating LGD R² distribution boxplot (HPO)...")
plot_performance_distribution(lgd_raw, 'R2', 'HPO', 'LGD')
plt.show()

### B2.3 Rank Distribution Boxplots

In [ ]:
print("Creating LGD R\u00b2 rank distribution boxplot (NO_HPO)...")
plot_rank_distribution(lgd_raw, 'R2', 'NO_HPO', 'LGD')
plt.show()

print("\nCreating LGD R\u00b2 rank distribution boxplot (HPO)...")
plot_rank_distribution(lgd_raw, 'R2', 'HPO', 'LGD')
plt.show()

## B3. Rank Analysis

### B3.1 Rank Heatmaps

In [ ]:
print("Creating LGD rank heatmap (NO_HPO)...")
fig_lgd_ranks_no_hpo, ranks_lgd_no_hpo, avg_ranks_lgd_no_hpo, median_ranks_lgd_no_hpo = create_rank_heatmap(
    lgd_agg, 'R2', 'NO_HPO', 'LGD'
)
plt.show()

print("\nMethod Rankings (NO_HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_lgd_no_hpo.items(), 1):
    med_rank = median_ranks_lgd_no_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

In [ ]:
print("Creating LGD rank heatmap (HPO)...")
fig_lgd_ranks_hpo, ranks_lgd_hpo, avg_ranks_lgd_hpo, median_ranks_lgd_hpo = create_rank_heatmap(
    lgd_agg, 'R2', 'HPO', 'LGD'
)
plt.show()

print("\nMethod Rankings (HPO):")
print(f"{'Rank':>5} {'Method':<20} {'Avg Rank':>10} {'Median Rank':>12}")
print("-" * 50)
for rank, (method, avg_rank) in enumerate(avg_ranks_lgd_hpo.items(), 1):
    med_rank = median_ranks_lgd_hpo[method]
    print(f"   {rank:2d}. {method:20s}  {avg_rank:>10.2f}  {med_rank:>12.2f}")

### B3.2 Average Rank Bar Charts

In [ ]:
if avg_ranks_lgd_no_hpo is not None:
    print("Creating LGD average rank bar chart (NO_HPO)...")
    plot_rank_bar(avg_ranks_lgd_no_hpo, median_ranks_lgd_no_hpo, 'LGD', 'R2', 'NO_HPO')
    plt.show()

if avg_ranks_lgd_hpo is not None:
    print("\nCreating LGD average rank bar chart (HPO)...")
    plot_rank_bar(avg_ranks_lgd_hpo, median_ranks_lgd_hpo, 'LGD', 'R2', 'HPO')
    plt.show()

## B4. Statistical Testing

### B4.1. PAMA Analysis

In [ ]:
print("\n" + "=" * 80)
print("  LGD - PAMA ANALYSIS (HPO) - Fold-Level")
print("=" * 80)

# Use raw data to calculate PAMA at fold level
lgd_hpo_raw = lgd_raw[lgd_raw['hpo_mode'] == 'HPO'].copy()

if not lgd_hpo_raw.empty and 'R2' in lgd_hpo_raw.columns:
    pama_scores = {}
    
    # Group by dataset and fold to find winner for each fold
    for (dataset, fold), group in lgd_hpo_raw.groupby(['dataset', 'fold_id']):
        if len(group) > 0 and 'R2' in group.columns:
            max_score = group['R2'].max()
            best_methods = group[group['R2'] >= max_score - 0.0001]['method'].tolist()
            
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))
    
    # Total number of folds
    n_folds = len(lgd_hpo_raw.groupby(['dataset', 'fold_id']))
    
    pama_df = pd.DataFrame([
        {'method': method, 'wins': score, 'pama_pct': 100 * score / n_folds}
        for method, score in pama_scores.items()
    ]).sort_values('pama_pct', ascending=False)
    
    print(f"\nPAMA Scores (across {n_folds} folds):\n")
    for idx, row in pama_df.iterrows():
        bar = '█' * int(row['pama_pct'] / 2)
        print(f"  {row['method']:20s}  {row['wins']:5.1f} wins  {row['pama_pct']:5.1f}%  {bar}")
    
    fig, ax = plt.subplots(figsize=(14, 8))
    colors = plt.cm.RdYlGn(pama_df['pama_pct'] / pama_df['pama_pct'].max())
    bars = ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=colors, edgecolor='black', linewidth=1.5)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])

    # Color foundation model labels red
    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')
    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'LGD: Probability of Achieving Maximum R² (HPO)\nFold-Level Analysis (n={n_folds} folds)', 
                 fontsize=16, fontweight='bold', pad=20)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()
    
    for i, (idx, row) in enumerate(pama_df.iterrows()):
        ax.text(row['pama_pct'] + 1, i, f"{row['pama_pct']:.1f}%", va='center', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    filename = FIGURES_DIR / "lgd_pama_analysis_hpo.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n✅ Saved: {filename.name}")
    plt.show()
else:
    print("⚠️  No HPO data available for PAMA analysis")

### B4.2 Full PAMA (All Methods)

In [ ]:
# Full PAMA including methods with 0% wins
print("\n" + "=" * 80)
print("  LGD - FULL PAMA ANALYSIS (HPO) - All Methods Including 0%")
print("=" * 80)

lgd_hpo_raw_full = lgd_raw[lgd_raw['hpo_mode'] == 'HPO'].copy()

if not lgd_hpo_raw_full.empty and 'R2' in lgd_hpo_raw_full.columns:
    pama_scores = {}

    for (dataset, fold), group in lgd_hpo_raw_full.groupby(['dataset', 'fold_id']):
        if len(group) > 0:
            max_score = group['R2'].max()
            best_methods = group[group['R2'] >= max_score - 0.0001]['method'].tolist()
            for method in best_methods:
                pama_scores[method] = pama_scores.get(method, 0) + (1.0 / len(best_methods))

    n_folds = len(lgd_hpo_raw_full.groupby(['dataset', 'fold_id']))
    all_methods = sorted(lgd_hpo_raw_full['method'].unique())

    pama_df = pd.DataFrame([
        {'method': m, 'wins': pama_scores.get(m, 0), 'pama_pct': 100 * pama_scores.get(m, 0) / n_folds}
        for m in all_methods
    ]).sort_values('pama_pct', ascending=False)

    fig, ax = plt.subplots(figsize=(14, max(8, len(pama_df) * 0.45)))

    bar_colors = []
    for _, row in pama_df.iterrows():
        if row['method'] in FOUNDATION_METHODS:
            bar_colors.append('indianred')
        elif row['pama_pct'] > 0:
            bar_colors.append('steelblue')
        else:
            bar_colors.append('lightgray')

    ax.barh(range(len(pama_df)), pama_df['pama_pct'], color=bar_colors, edgecolor='black', linewidth=0.8, alpha=0.85)
    ax.set_yticks(range(len(pama_df)))
    ax.set_yticklabels(pama_df['method'])

    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')

    ax.set_xlabel('PAMA (%)', fontsize=14, fontweight='bold')
    ax.set_title(f'LGD: PAMA (R²) - All Methods (HPO)\nn={n_folds} folds | Red bars = Foundation Models | Gray = Never Best',
                 fontsize=13, fontweight='bold', pad=15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    ax.invert_yaxis()

    for i, (idx, row) in enumerate(pama_df.iterrows()):
        label = f"{row['pama_pct']:.1f}%" if row['pama_pct'] > 0 else "0%"
        ax.text(max(row['pama_pct'] + 0.5, 1.5), i, label, va='center', fontsize=9, fontweight='bold')

    plt.tight_layout()
    filename = FIGURES_DIR / "lgd_pama_full_all_methods.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\n\u2705 Saved: {filename.name}")
    plt.show()


### B4.3 Foundation Model Spotlight

In [ ]:
# Foundation Model Spotlight for LGD
print("\n" + "=" * 80)
print("  LGD - FOUNDATION MODEL SPOTLIGHT")
print("=" * 80)

lgd_hpo_raw_spot = lgd_raw[lgd_raw['hpo_mode'] == 'HPO'].copy()

if not lgd_hpo_raw_spot.empty and 'R2' in lgd_hpo_raw_spot.columns:
    # --- 1. Average R2: Foundation vs Non-Foundation ---
    avg_r2 = lgd_hpo_raw_spot.groupby('method')['R2'].mean().reset_index()
    avg_r2['is_foundation'] = avg_r2['method'].isin(FOUNDATION_METHODS)
    avg_r2 = avg_r2.sort_values('R2', ascending=True)

    fig, ax = plt.subplots(figsize=(12, max(6, len(avg_r2) * 0.4)))
    bar_colors = ['indianred' if f else 'steelblue' for f in avg_r2['is_foundation']]
    ax.barh(range(len(avg_r2)), avg_r2['R2'], color=bar_colors, edgecolor='black', linewidth=0.5, alpha=0.85)
    ax.set_yticks(range(len(avg_r2)))
    ax.set_yticklabels(avg_r2['method'])
    for label in ax.get_yticklabels():
        if label.get_text() in FOUNDATION_METHODS:
            label.set_color('red')
            label.set_fontweight('bold')

    ax.set_xlabel('Average R\u00b2 (across all datasets & folds)', fontweight='bold')
    ax.set_title('LGD: Average R\u00b2 per Method (HPO)\nRed = Foundation Models', fontsize=13, fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    for i, (_, row) in enumerate(avg_r2.iterrows()):
        ax.text(row['R2'] + 0.005, i, f"{row['R2']:.4f}", va='center', fontsize=8)

    plt.tight_layout()
    filename = FIGURES_DIR / "lgd_foundation_spotlight_avg_r2.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\u2705 Saved: {filename.name}")
    plt.show()

    # --- 2. Win rate ---
    print("\nFoundation vs Non-Foundation Win Rate (per fold):")
    fm_wins = 0
    nfm_wins = 0
    ties = 0
    n_total = 0

    for (dataset, fold), group in lgd_hpo_raw_spot.groupby(['dataset', 'fold_id']):
        n_total += 1
        fm_group = group[group['method'].isin(FOUNDATION_METHODS)]
        nfm_group = group[~group['method'].isin(FOUNDATION_METHODS)]

        if fm_group.empty or nfm_group.empty:
            continue

        best_fm = fm_group['R2'].max()
        best_nfm = nfm_group['R2'].max()

        if best_fm > best_nfm + 0.0001:
            fm_wins += 1
        elif best_nfm > best_fm + 0.0001:
            nfm_wins += 1
        else:
            ties += 1

    labels = ['Foundation\nModel Wins', 'Non-Foundation\nWins', 'Ties']
    counts = [fm_wins, nfm_wins, ties]
    colors_pie = ['indianred', 'steelblue', 'lightgray']

    fig, ax = plt.subplots(figsize=(8, 8))
    wedges, texts, autotexts = ax.pie(counts, labels=labels, colors=colors_pie,
                                       autopct='%1.1f%%', startangle=90,
                                       textprops={'fontsize': 12, 'fontweight': 'bold'})
    for autotext in autotexts:
        autotext.set_fontsize(13)
        autotext.set_fontweight('bold')
    ax.set_title(f'LGD: Foundation vs Non-Foundation Win Rate (HPO)\n{n_total} total folds',
                 fontsize=13, fontweight='bold', pad=20)

    plt.tight_layout()
    filename = FIGURES_DIR / "lgd_foundation_vs_nonfoundation_winrate.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"\u2705 Saved: {filename.name}")
    plt.show()

    print(f"\n  Foundation model best: {fm_wins}/{n_total} ({100*fm_wins/n_total:.1f}%)")
    print(f"  Non-foundation best:  {nfm_wins}/{n_total} ({100*nfm_wins/n_total:.1f}%)")
    print(f"  Ties:                 {ties}/{n_total} ({100*ties/n_total:.1f}%)")


### B4.4 Pairwise Statistical Comparisons & Critical Difference Diagram

Friedman omnibus test, pairwise paired T-tests with Holm correction, and Critical Difference diagram (Demsar, 2006) for LGD methods.

In [ ]:
print("\n" + "=" * 80)
print("  LGD - PAIRWISE STATISTICAL COMPARISONS (HPO)")
print("=" * 80)

# Build performance matrix: (dataset x fold) x method
lgd_perf = compute_performance_matrix(lgd_raw, 'R2', hpo_mode='HPO')

if lgd_perf is not None and lgd_perf.shape[1] >= 3:
    print(f"\nPerformance matrix: {lgd_perf.shape[0]} observations x {lgd_perf.shape[1]} methods")

    # --- Friedman Test ---
    f_stat, f_p = friedman_test(lgd_perf)
    print("\n" + "\u2500" * 60)
    print(f"Friedman test:  \u03c7\u00b2 = {f_stat:.2f},  p = {f_p:.2e}")
    if f_p < 0.05:
        print("  -> Significant differences exist among methods (p < 0.05)")
    else:
        print("  -> No significant differences detected (p >= 0.05)")
    print("\u2500" * 60)

    # --- Average Ranks ---
    lgd_avg_ranks = compute_average_ranks(lgd_perf, higher_is_better=True)
    print(f"\nAverage Ranks (1 = best):")
    for method, rank in lgd_avg_ranks.items():
        marker = " *" if method in FOUNDATION_METHODS else ""
        print(f"  {method:20s}  {rank:.2f}{marker}")

    # --- Pairwise T-tests with Holm correction ---
    lgd_pairwise = pairwise_ttest_holm(lgd_perf)
    n_sig = lgd_pairwise['significant'].sum()
    n_total = len(lgd_pairwise)
    print(f"\nPairwise T-tests (Holm correction):")
    print(f"  {n_sig}/{n_total} pairs significantly different (alpha=0.05)")

    sig_pairs = lgd_pairwise[lgd_pairwise['significant']].sort_values('p_corrected')
    if len(sig_pairs) > 0:
        print(f"\n  Top significant pairs:")
        for _, row in sig_pairs.head(15).iterrows():
            print(f"    {row['method_a']:18s} vs {row['method_b']:18s}  p={row['p_corrected']:.4f}")

    # --- Critical Difference Diagram ---
    print("\n" + "\u2500" * 60)
    print("Generating Critical Difference diagram...")
    plot_critical_difference_diagram(
        lgd_avg_ranks, lgd_pairwise,
        title=f'LGD: Critical Difference Diagram (R2, HPO)\nFriedman p={f_p:.2e}  |  {lgd_perf.shape[0]} observations',
        figsize=(18, max(10, len(lgd_avg_ranks) * 0.45)),
        foundation_methods=FOUNDATION_METHODS,
        save_path=FIGURES_DIR / "lgd_critical_difference_r2_hpo.png"
    )

    # --- Pairwise Significance Heatmap ---
    print("Generating pairwise significance heatmap...")
    plot_pairwise_significance_heatmap(
        lgd_pairwise, lgd_avg_ranks,
        title='LGD: Pairwise T-Test p-values (Holm-corrected, R2 HPO)',
        save_path=FIGURES_DIR / "lgd_pairwise_significance_heatmap_r2_hpo.png"
    )

    # Horizontal variant
    plot_critical_difference_diagram(
        lgd_avg_ranks, lgd_pairwise,
        title=f'LGD: Critical Difference Diagram (R2, HPO)\nFriedman p={f_p:.2e}',
        figsize=(20, max(10, len(lgd_avg_ranks) * 0.45)),
        foundation_methods=FOUNDATION_METHODS,
        save_path=FIGURES_DIR / "lgd_critical_difference_r2_hpo_horizontal.png"
    )
else:
    print("Warning: Insufficient data for LGD statistical analysis")

### B4.5 PAMA Analysis (Probability of Achieving Maximum Accuracy)

In [ ]:
print("\n" + "=" * 80)
print("  LGD - PAMA ANALYSIS (Probability of Achieving Maximum Accuracy)")
print("=" * 80)

for hpo_mode in ['HPO', 'NO_HPO']:
    lgd_perf_pama = compute_performance_matrix(lgd_raw, 'R2', hpo_mode=hpo_mode)
    if lgd_perf_pama is not None:
        pama = compute_pama(lgd_perf_pama, higher_is_better=True)
        n_obs = len(lgd_perf_pama)
        print(f"\nPAMA - LGD R2 ({hpo_mode}): {n_obs} observations, {len(lgd_perf_pama.columns)} methods")
        print("-" * 60)
        for method, frac in pama.items():
            marker = " *" if method in FOUNDATION_METHODS else ""
            print(f"  {method:25s}  {frac*100:5.1f}%  ({int(frac*n_obs):3d}/{n_obs} wins){marker}")

        fm_wins = sum(frac for m, frac in pama.items() if m in FOUNDATION_METHODS)
        print(f"\n  Foundation models (group): {fm_wins*100:.1f}%")

        plot_pama(
            pama,
            title=f'LGD: PAMA Analysis (R2, {hpo_mode})\n{n_obs} observations',
            foundation_methods=FOUNDATION_METHODS,
            figsize=(14, max(6, sum(1 for v in pama.values if v > 0) * 0.45)),
            save_path=FIGURES_DIR / f"lgd_pama_r2_{hpo_mode.lower()}.png"
        )

---
# 📐 Part C: Dataset Characteristics Analysis

Analyze the relationship between method rankings and dataset characteristics:
- **Dataset size** (number of rows)
- **Number of features** (columns)
- **Dimensionality** (rows × columns)

## C1. Load Dataset Characteristics

In [ ]:
print("\n" + "=" * 80)
print("  LOADING DATASET CHARACTERISTICS")
print("=" * 80)

# Find processed data directory
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

# DataFeeder PCA settings (from data_feeder.py)
MAX_FEATURES_THRESHOLD = 100  # PCA triggered if features exceed this
PCA_TARGET_FEATURES = 99       # Target number of PCA components

print(f"\n📂 Looking for processed data in: {PROCESSED_DATA_DIR}")

if not PROCESSED_DATA_DIR.exists():
    print(f"⚠️  Processed data directory not found: {PROCESSED_DATA_DIR}")
    print("   Run preprocessing first to generate processed datasets!")
else:
    print(f"✓ Found processed data directory")

# Collect dataset characteristics
dataset_info = []

# Get unique datasets from results
all_datasets_pd = pd_raw['dataset'].unique()
all_datasets_lgd = lgd_raw['dataset'].unique()

print(f"\n📊 Extracting characteristics for {len(all_datasets_pd)} PD datasets and {len(all_datasets_lgd)} LGD datasets...")

# Function to get dataset characteristics
def get_dataset_characteristics(dataset_name, task):
    """
    Extract dataset characteristics from processed numpy arrays.
    
    Reflects the ACTUAL features used during training:
    - If total features > 100, PCA reduces to 99 components
    - Otherwise, uses original feature count
    
    The preprocessing pipeline stores datasets as:
    data/processed/{task}/{dataset}/
        ├── N.npy (numerical features) - shape: (n_samples, n_num_features)
        ├── C.npy (categorical features) - shape: (n_samples, n_cat_features)  
        ├── y.npy (target) - shape: (n_samples,)
        └── info.json (metadata)
    """
    if not PROCESSED_DATA_DIR.exists():
        return None
    
    # Build path: data/processed/{task}/{dataset}/
    task_dir = PROCESSED_DATA_DIR / task.lower() / dataset_name
    
    # Check if directory exists
    if not task_dir.exists():
        return None
    
    # Check if required files exist
    y_path = task_dir / "y.npy"
    if not y_path.exists():
        return None
    
    try:
        # Load target array to get number of rows and statistics
        y = np.load(y_path, allow_pickle=False)
        n_rows = len(y)
        
        # --- NEW: Calculate Class Imbalance for PD ---
        pos_class_rate = np.nan
        if task == 'PD':
            # Assuming binary classification where 1 is the default class
            # Calculate mean (proportion of 1s)
            pos_class_rate = np.mean(y)
        
        # Count numerical features
        n_num_features = 0
        N_path = task_dir / "N.npy"
        if N_path.exists():
            N = np.load(N_path, allow_pickle=False)
            if N.ndim == 2:
                n_num_features = N.shape[1]
            elif N.ndim == 1:
                n_num_features = 1
        
        # Count categorical features
        n_cat_features = 0
        C_path = task_dir / "C.npy"
        if C_path.exists():
            C = np.load(C_path, allow_pickle=False)
            if C.ndim == 2:
                n_cat_features = C.shape[1]
            elif C.ndim == 1:
                n_cat_features = 1
        
        # Total features BEFORE PCA
        n_features_raw = n_num_features + n_cat_features
        
        # Apply DataFeeder logic: if features > threshold, PCA reduces to target
        if n_features_raw > MAX_FEATURES_THRESHOLD:
            n_features_actual = PCA_TARGET_FEATURES
            pca_applied = True
        else:
            n_features_actual = n_features_raw
            pca_applied = False
        
        return {
            'dataset': dataset_name,
            'task': task,
            'n_rows': n_rows,
            'n_features': n_features_actual,  # Features AFTER potential PCA
            'n_features_raw': n_features_raw, # Features BEFORE PCA
            'n_num_features': n_num_features,
            'n_cat_features': n_cat_features,
            'pos_class_rate': pos_class_rate, # New characteristic: % of defaults
            'pca_applied': pca_applied,
            'dimensionality': n_rows * n_features_actual,
            'source': 'numpy_arrays'
        }
        
    except Exception as e:
        print(f"⚠️  Error loading numpy arrays for {dataset_name}: {e}")
        return None

# Collect info for all datasets
print("\n🔍 Processing datasets...")
found_count = 0
missing_count = 0

# --- Process PD ---
for dataset in all_datasets_pd:
    info = get_dataset_characteristics(dataset, 'PD')
    if info:
        dataset_info.append(info)
        found_count += 1
        pca_note = " [PCA→99]" if info['pca_applied'] else ""
        imbalance_str = f", {info['pos_class_rate']:.1%} defaults"
        print(f"  ✓ {dataset:30s} - {info['n_rows']:>8,} rows, {info['n_features']:>3} features ({info['n_num_features']} num + {info['n_cat_features']} cat){pca_note}{imbalance_str}")
    else:
        missing_count += 1
        print(f"  ✗ {dataset:30s} - NOT FOUND in processed directory")

# --- Process LGD ---
for dataset in all_datasets_lgd:
    info = get_dataset_characteristics(dataset, 'LGD')
    if info:
        dataset_info.append(info)
        found_count += 1
        pca_note = " [PCA→99]" if info['pca_applied'] else ""
        print(f"  ✓ {dataset:30s} - {info['n_rows']:>8,} rows, {info['n_features']:>3} features ({info['n_num_features']} num + {info['n_cat_features']} cat){pca_note}")
    else:
        missing_count += 1
        print(f"  ✗ {dataset:30s} - NOT FOUND in processed directory")

# Create DataFrame
if dataset_info:
    dataset_chars = pd.DataFrame(dataset_info)
    
    # Drop internal columns from display
    display_df = dataset_chars.drop(columns=['source', 'n_num_features', 'n_cat_features', 'n_features_raw', 'pca_applied'], errors='ignore')
    
    # Rename column for clarity
    display_df = display_df.rename(columns={'pos_class_rate': 'default_rate'})
    
    print(f"\n{'='*80}")
    print(f"✅ Successfully loaded {found_count} datasets")
    if missing_count > 0:
        print(f"⚠️  {missing_count} datasets not found (need preprocessing)")
    print(f"{'='*80}")
    
    # Show how many datasets had PCA applied
    n_pca = dataset_chars['pca_applied'].sum()
    if n_pca > 0:
        print(f"\n📊 {n_pca} datasets will use PCA (features > {MAX_FEATURES_THRESHOLD} → reduced to {PCA_TARGET_FEATURES})")
    
    print(f"\nDataset characteristics summary:")
    print(display_df[['n_rows', 'n_features', 'dimensionality']].describe())
    
    print(f"\nAll datasets (with default rates for PD):")
    # Format the default_rate column to percentage string for display
    display_df['default_rate'] = display_df['default_rate'].apply(lambda x: f"{x:.2%}" if pd.notnull(x) else "-")
    print(display_df.to_string(index=False))
    
else:
    dataset_chars = pd.DataFrame()
    print(f"\n{'='*80}")
    print("⚠️  NO datasets found in processed directory!")
    print(f"{'='*80}")
    print("\n💡 Next steps:")
    print("   1. Make sure datasets have been preprocessed")
    print("   2. Check that processed files exist in:")
    print(f"      {PROCESSED_DATA_DIR}/{{task}}/{{dataset}}/")
    print("   3. Each dataset folder should contain: N.npy, C.npy, y.npy")

## C2. PD: Rank vs Dataset Characteristics

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

print("\n" + "=" * 80)
print("  PD - RANK CORRELATION WITH DATASET CHARACTERISTICS")
print("=" * 80)

if not dataset_chars.empty and pivot_pd_hpo is not None:
    # Calculate ranks for each method across datasets
    ranks_df = pivot_pd_hpo.rank(axis=1, ascending=False, method='average')

    # Get PD datasets
    pd_chars = dataset_chars[dataset_chars['task'] == 'PD'].copy()

    # Calculate class imbalance (minority class proportion)
    if 'pos_class_rate' in pd_chars.columns:
        pd_chars['class_imbalance'] = pd_chars['pos_class_rate'].apply(
            lambda x: min(x, 1-x) if pd.notna(x) else np.nan
        )

    # Merge with dataset characteristics
    pd_chars = pd_chars.set_index('dataset')

    # Calculate correlations for each method
    correlations = []

    for method in ranks_df.columns:
        method_ranks = ranks_df[method]
        common_datasets = method_ranks.index.intersection(pd_chars.index)

        if len(common_datasets) > 3:
            aligned_ranks = method_ranks.loc[common_datasets]
            aligned_chars = pd_chars.loc[common_datasets]

            if aligned_chars['n_rows'].notna().sum() > 3:
                corr_size, p_size = stats.spearmanr(aligned_ranks, aligned_chars['n_rows'].fillna(aligned_chars['n_rows'].median()))
            else:
                corr_size, p_size = np.nan, np.nan

            if aligned_chars['n_features'].notna().sum() > 3:
                corr_features, p_features = stats.spearmanr(aligned_ranks, aligned_chars['n_features'].fillna(aligned_chars['n_features'].median()))
            else:
                corr_features, p_features = np.nan, np.nan

            if 'class_imbalance' in aligned_chars.columns and aligned_chars['class_imbalance'].notna().sum() > 3:
                corr_imbal, p_imbal = stats.spearmanr(aligned_ranks, aligned_chars['class_imbalance'].fillna(aligned_chars['class_imbalance'].median()))
            else:
                corr_imbal, p_imbal = np.nan, np.nan

            correlations.append({
                'method': method,
                'corr_size': corr_size, 'p_size': p_size,
                'corr_features': corr_features, 'p_features': p_features,
                'corr_imbalance': corr_imbal, 'p_imbalance': p_imbal
            })

    corr_df = pd.DataFrame(correlations)

    if not corr_df.empty:
        print("\n\U0001f4ca Spearman Correlations (Rank vs Dataset Characteristics)")
        print("   Positive correlation = higher rank (worse) as metric increases")
        print("   Negative correlation = lower rank (better) as metric increases\n")

        print(f"{'Method':<20} {'Size':>9} {'p-val':>7} {'Features':>9} {'p-val':>7} {'Imbalance':>9} {'p-val':>7}")
        print("-" * 80)

        for _, row in corr_df.iterrows():
            sig_size = "*" if row['p_size'] < 0.05 else " "
            sig_feat = "*" if row['p_features'] < 0.05 else " "
            sig_imbal = "*" if row['p_imbalance'] < 0.05 else " "

            print(f"{row['method']:<20} "
                  f"{row['corr_size']:>8.3f}{sig_size} {row['p_size']:>7.3f} "
                  f"{row['corr_features']:>8.3f}{sig_feat} {row['p_features']:>7.3f} "
                  f"{row['corr_imbalance']:>8.3f}{sig_imbal} {row['p_imbalance']:>7.3f}")

        # =======================================================================
        # VISUALIZATION: 1x3 Grid of Bar Charts (removed dimensionality)
        # =======================================================================
        fig, axes = plt.subplots(1, 3, figsize=(24, 8))

        def plot_correlation_bar(ax, data, col_name, title):
            corr_sorted = data.sort_values(col_name, ascending=False)
            colors = ['red' if x > 0 else 'green' for x in corr_sorted[col_name]]
            ax.barh(range(len(corr_sorted)), corr_sorted[col_name], color=colors, alpha=0.7, edgecolor='black')
            ax.set_yticks(range(len(corr_sorted)))
            ax.set_yticklabels(corr_sorted['method'], fontsize=15)
            ax.set_xlabel('Spearman Correlation', fontweight='bold', fontsize=15)
            ax.set_title(title, fontweight='bold', fontsize=20, pad=10)
            ax.axvline(0, color='black', linewidth=1.5)
            ax.grid(axis='x', alpha=0.3)

            # Color foundation model labels red
            for label in ax.get_yticklabels():
                if label.get_text() in FOUNDATION_METHODS:
                    label.set_color('red')
                    label.set_fontweight('bold')

            for i, (idx, row) in enumerate(corr_sorted.iterrows()):
                val = row[col_name]
                if pd.notna(val):
                    ax.text(val + 0.02 if val > 0 else val - 0.02, i, f'{val:.2f}',
                           va='center', ha='left' if val > 0 else 'right', fontsize=8)

        plot_correlation_bar(axes[0], corr_df, 'corr_size',
                             'Rank vs Dataset Size\n(red=worse on large, green=better)')

        plot_correlation_bar(axes[1], corr_df, 'corr_features',
                             'Rank vs Feature Count\n(red=worse on high-dim, green=better)')

        plot_correlation_bar(axes[2], corr_df, 'corr_imbalance',
                             'Rank vs Class Imbalance\n(red=worse when imbalanced, green=better)')

        plt.suptitle('PD: Rank Correlation with Dataset Characteristics (HPO)',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        filename = FIGURES_DIR / "pd_rank_correlation_dataset_chars.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"\n\u2705 Saved: {filename.name}")
        plt.show()

    else:
        print("\u26a0\ufe0f  Not enough data for correlation analysis")
else:
    print("\u26a0\ufe0f  Dataset characteristics or HPO results not available")

## C3. PD: Rank vs Dataset Characteristics per method

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats

# ==============================================================================
# CONFIGURATION
# ==============================================================================
METHODS_TO_ANALYZE = ['tabpfn', 'tabpfn_v2', 'catboost', 'tabicl']

print("\n" + "="*80)
print("  PD - RANK SCATTER PLOTS (Dataset-Level Ranks)")
print("="*80)

if 'pd_agg' not in locals() or 'dataset_chars' not in locals():
    print("⚠️  Error: 'pd_agg' or 'dataset_chars' not found. Please run the previous data loading cells.")
else:
    # ==============================================================================
    # USE THE EXACT SAME RANKING METHODOLOGY AS THE HEATMAP
    # ==============================================================================
    
    # 1. Filter for HPO mode only (same as heatmap)
    pd_agg_hpo = pd_agg[pd_agg['hpo_mode'] == 'HPO'].copy()
    
    # 2. Identify the metric column
    metric_col = 'AUC_mean'  # Aggregated mean AUC per dataset
    
    if metric_col not in pd_agg_hpo.columns:
        print(f"⚠️  Error: '{metric_col}' not found in pd_agg")
        print(f"   Available columns: {list(pd_agg_hpo.columns)}")
    else:
        print(f"✓ Using aggregated data with metric: '{metric_col}'")
        
        # 3. Create pivot table (same as heatmap)
        pivot = pd_agg_hpo.pivot(index='dataset', columns='method', values=metric_col)
        
        print(f"✓ Pivot table shape: {pivot.shape}")
        print(f"  - Datasets: {len(pivot)}")
        print(f"  - Methods: {len(pivot.columns)}")
        
        # 4. Calculate ranks (EXACT SAME as heatmap: rank across methods for each dataset)
        ranks_df = pivot.rank(axis=1, ascending=False, method='average')
        
        # 5. Convert ranks to long format for plotting
        ranks_long = ranks_df.reset_index().melt(
            id_vars='dataset', 
            var_name='method', 
            value_name='rank'
        )
        
        print(f"✓ Ranks calculated for {len(ranks_long)} dataset-method combinations")
        
        # 6. Prepare dataset characteristics
        pd_chars = dataset_chars[dataset_chars['task'] == 'PD'].copy()
        
        # Calculate class imbalance (minority class proportion)
        if 'pos_class_rate' in pd_chars.columns:
            pd_chars['class_imbalance'] = pd_chars['pos_class_rate'].apply(
                lambda x: min(x, 1-x) if pd.notna(x) else np.nan
            )
        
        # 7. Merge ranks with dataset characteristics
        plot_data = pd.merge(ranks_long, pd_chars, on='dataset', how='inner')
        
        print(f"✓ Merged data: {len(plot_data)} dataset-method combinations")
        print(f"  - Datasets with characteristics: {plot_data['dataset'].nunique()}")
        
        # Filter for the specific methods requested
        valid_methods = [m for m in METHODS_TO_ANALYZE if m in plot_data['method'].unique()]
        print(f"✓ Methods to plot: {valid_methods}\n")
        
        # Print diagnostic info for each method
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method]
            if not method_data.empty:
                print(f"  {method}:")
                print(f"    - Datasets: {len(method_data)}")
                print(f"    - Rank range: {method_data['rank'].min():.2f} - {method_data['rank'].max():.2f}")
                print(f"    - Rank 1 count: {(method_data['rank'] == 1.0).sum()} datasets")
                print(f"    - Average rank: {method_data['rank'].mean():.2f}")
        
        # ==============================================================================
        # GENERATE PLOTS
        # ==============================================================================
        sns.set_style("whitegrid")
        
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method].copy()
            if method_data.empty: 
                print(f"⚠️  No data for method: {method}")
                continue
            
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(f"Method: {method} - Dataset-Level Rank (Lower is Better)", 
                        fontsize=18, fontweight='bold', y=0.98)
            
            # Helper for clean plotting
            def plot_clean(ax, x_col, label, log_scale=False):
                if x_col not in method_data.columns: 
                    ax.text(0.5, 0.5, f'No {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Remove NaN values for this specific plot
                plot_subset = method_data[[x_col, 'rank', 'dataset']].dropna()
                
                if len(plot_subset) == 0:
                    ax.text(0.5, 0.5, f'No valid {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Scatter: Blue dots
                scatter = ax.scatter(
                    plot_subset[x_col], plot_subset['rank'],
                    color='tab:blue', s=80, alpha=0.7, edgecolor='darkblue', linewidth=1.5
                )
                
                # Trendline
                try:
                    sns.regplot(
                        data=plot_subset, x=x_col, y='rank',
                        ax=ax, scatter=False, color='red',
                        line_kws={'linestyle': '--', 'linewidth': 2, 'alpha': 0.7}
                    )
                except Exception as e:
                    print(f"  ⚠️  Could not fit trendline for {label}: {e}")
                
                # Formatting
                ax.set_xlabel(label, fontsize=12, fontweight='bold')
                ax.set_ylabel('Rank (Lower = Better)', fontsize=12, fontweight='bold')
                
                # Y-axis: Start from 0.5, go up to max rank + 0.5
                y_max = plot_data['rank'].max() + 0.5
                ax.set_ylim(0.5, y_max)
                
                # Log scale for x if requested
                if log_scale:
                    ax.set_xscale('log')
                
                # Grid
                ax.grid(True, linestyle=':', alpha=0.4, linewidth=0.8)

            # Plot 1: Class Imbalance (Top Left)
            if 'class_imbalance' in method_data.columns:
                plot_clean(axes[0], 'class_imbalance', 'Class Imbalance (Minority Proportion)', log_scale=False)
                axes[0].xaxis.set_major_formatter(FuncFormatter(lambda x, _: f'{x:.0%}'))
                axes[0].set_title("Rank vs Class Imbalance", fontsize=13, pad=10, fontweight='bold')
            else:
                axes[0].text(0.5, 0.5, 'No class imbalance data', ha='center', va='center', 
                              transform=axes[0].transAxes, fontsize=12)

            # Plot 2: Rows (Top Right) - Log scale
            plot_clean(axes[1], 'n_rows', 'Number of Rows (Log Scale)', log_scale=True)
            axes[1].set_title("Rank vs Dataset Size", fontsize=13, pad=10, fontweight='bold')

            # Plot 3: Features (Bottom Left) - Log scale
            plot_clean(axes[2], 'n_features', 'Number of Features (Log Scale)', log_scale=True)
            axes[2].set_title("Rank vs Feature Count", fontsize=13, pad=10, fontweight='bold')


            plt.tight_layout(rect=[0, 0, 1, 0.98])
            
            filename = FIGURES_DIR / f"pd_rank_scatter_{method}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"\n✅ Saved: {filename.name}")
            plt.show()
            
        print("\n" + "="*80)
        print("✅ All scatter plots generated successfully!")
        print("  These plots now use EXACTLY the same ranking as the heatmap:")
        print("  - Ranks based on mean AUC per dataset")
        print("  - Each dot = one dataset")
        print("  - Rank 1 dots should match heatmap exactly")
        print("="*80)

## C4. LGD: Rank vs Dataset Characteristics

In [ ]:
print("\n" + "=" * 80)
print("  LGD - RANK CORRELATION WITH DATASET CHARACTERISTICS")
print("=" * 80)

if not dataset_chars.empty and pivot_lgd_hpo is not None:
    # Calculate ranks for each method across datasets
    ranks_df = pivot_lgd_hpo.rank(axis=1, ascending=False, method='average')

    # Get LGD datasets
    lgd_chars = dataset_chars[dataset_chars['task'] == 'LGD'].copy()
    lgd_chars = lgd_chars.set_index('dataset')

    correlations = []

    for method in ranks_df.columns:
        method_ranks = ranks_df[method]
        common_datasets = method_ranks.index.intersection(lgd_chars.index)

        if len(common_datasets) > 3:
            aligned_ranks = method_ranks.loc[common_datasets]
            aligned_chars = lgd_chars.loc[common_datasets]

            if aligned_chars['n_rows'].notna().sum() > 3:
                corr_size, p_size = stats.spearmanr(aligned_ranks, aligned_chars['n_rows'].fillna(aligned_chars['n_rows'].median()))
            else:
                corr_size, p_size = np.nan, np.nan

            if aligned_chars['n_features'].notna().sum() > 3:
                corr_features, p_features = stats.spearmanr(aligned_ranks, aligned_chars['n_features'].fillna(aligned_chars['n_features'].median()))
            else:
                corr_features, p_features = np.nan, np.nan

            correlations.append({
                'method': method,
                'corr_size': corr_size, 'p_size': p_size,
                'corr_features': corr_features, 'p_features': p_features,
            })

    corr_df = pd.DataFrame(correlations)

    if not corr_df.empty:
        print("\n\U0001f4ca Spearman Correlations (Rank vs Dataset Characteristics)")
        print("   Positive correlation = higher rank (worse) on larger/more complex datasets")
        print("   Negative correlation = lower rank (better) on larger/more complex datasets\n")

        print(f"{'Method':<20} {'Size':>10} {'p-val':>8} {'Features':>10} {'p-val':>8}")
        print("-" * 60)

        for _, row in corr_df.iterrows():
            sig_size = "*" if row['p_size'] < 0.05 else " "
            sig_feat = "*" if row['p_features'] < 0.05 else " "

            print(f"{row['method']:<20} {row['corr_size']:>9.3f}{sig_size} {row['p_size']:>8.3f} "
                  f"{row['corr_features']:>9.3f}{sig_feat} {row['p_features']:>8.3f}")

        # Create visualization (1x2: Size and Features only)
        fig, axes = plt.subplots(1, 2, figsize=(20, 8))

        def plot_corr_bar_lgd(ax, data, col_name, title):
            corr_sorted = data.sort_values(col_name, ascending=False)
            colors = ['red' if x > 0 else 'green' for x in corr_sorted[col_name]]
            ax.barh(range(len(corr_sorted)), corr_sorted[col_name], color=colors, alpha=0.7, edgecolor='black')
            ax.set_yticks(range(len(corr_sorted)))
            ax.set_yticklabels(corr_sorted['method'])
            ax.set_xlabel('Spearman Correlation', fontweight='bold')
            ax.set_title(title, fontweight='bold')
            ax.axvline(0, color='black', linewidth=1)
            ax.grid(axis='x', alpha=0.3)

            # Color foundation model labels red
            for label in ax.get_yticklabels():
                if label.get_text() in FOUNDATION_METHODS:
                    label.set_color('red')
                    label.set_fontweight('bold')

        plot_corr_bar_lgd(axes[0], corr_df, 'corr_size',
                          'LGD: Rank vs Dataset Size\n(red=worse on large, green=better on large)')

        plot_corr_bar_lgd(axes[1], corr_df, 'corr_features',
                          'LGD: Rank vs Number of Features\n(red=worse on high-dim, green=better on high-dim)')

        plt.suptitle('LGD: Rank Correlation with Dataset Characteristics (HPO)',
                     fontsize=14, fontweight='bold', y=1.02)
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_rank_correlation_dataset_chars.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"\n\u2705 Saved: {filename.name}")
        plt.show()
    else:
        print("\u26a0\ufe0f  Not enough data for correlation analysis")
else:
    print("\u26a0\ufe0f  Dataset characteristics or HPO results not available")

## C5. LGD: Rank vs Dataset Characteristics per method

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from scipy import stats

# ==============================================================================
# CONFIGURATION
# ==============================================================================
METHODS_TO_ANALYZE = ['tabpfn', 'tabpfn_v2', 'catboost', 'tabicl']

print("\n" + "="*80)
print("  LGD - RANK SCATTER PLOTS (Dataset-Level Ranks)")
print("="*80)

if 'lgd_agg' not in locals() or 'dataset_chars' not in locals():
    print("⚠️  Error: 'lgd_agg' or 'dataset_chars' not found. Please run the previous data loading cells.")
else:
    # ==============================================================================
    # USE THE EXACT SAME RANKING METHODOLOGY AS THE HEATMAP
    # ==============================================================================
    
    # 1. Filter for HPO mode only (same as heatmap)
    lgd_agg_hpo = lgd_agg[lgd_agg['hpo_mode'] == 'HPO'].copy()
    
    # 2. Identify the metric column
    metric_col = 'R2_mean'  # Aggregated mean R² per dataset
    
    if metric_col not in lgd_agg_hpo.columns:
        print(f"⚠️  Error: '{metric_col}' not found in lgd_agg")
        print(f"   Available columns: {list(lgd_agg_hpo.columns)}")
    else:
        print(f"✓ Using aggregated data with metric: '{metric_col}'")
        
        # 3. Create pivot table (same as heatmap)
        pivot = lgd_agg_hpo.pivot(index='dataset', columns='method', values=metric_col)
        
        print(f"✓ Pivot table shape: {pivot.shape}")
        print(f"  - Datasets: {len(pivot)}")
        print(f"  - Methods: {len(pivot.columns)}")
        
        # 4. Calculate ranks (EXACT SAME as heatmap: rank across methods for each dataset)
        ranks_df = pivot.rank(axis=1, ascending=False, method='average')
        
        # 5. Convert ranks to long format for plotting
        ranks_long = ranks_df.reset_index().melt(
            id_vars='dataset', 
            var_name='method', 
            value_name='rank'
        )
        
        print(f"✓ Ranks calculated for {len(ranks_long)} dataset-method combinations")
        
        # 6. Prepare dataset characteristics (LGD only)
        lgd_chars = dataset_chars[dataset_chars['task'] == 'LGD'].copy()
        
        # 7. Merge ranks with dataset characteristics
        plot_data = pd.merge(ranks_long, lgd_chars, on='dataset', how='inner')
        
        print(f"✓ Merged data: {len(plot_data)} dataset-method combinations")
        print(f"  - Datasets with characteristics: {plot_data['dataset'].nunique()}")
        
        # Filter for the specific methods requested
        valid_methods = [m for m in METHODS_TO_ANALYZE if m in plot_data['method'].unique()]
        print(f"✓ Methods to plot: {valid_methods}\n")
        
        # Print diagnostic info for each method
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method]
            if not method_data.empty:
                print(f"  {method}:")
                print(f"    - Datasets: {len(method_data)}")
                print(f"    - Rank range: {method_data['rank'].min():.2f} - {method_data['rank'].max():.2f}")
                print(f"    - Rank 1 count: {(method_data['rank'] == 1.0).sum()} datasets")
                print(f"    - Average rank: {method_data['rank'].mean():.2f}")
        
        # ==============================================================================
        # GENERATE PLOTS (3 plots: rows, features, dimensionality - NO class imbalance)
        # ==============================================================================
        sns.set_style("whitegrid")
        
        for method in valid_methods:
            method_data = plot_data[plot_data['method'] == method].copy()
            if method_data.empty: 
                print(f"⚠️  No data for method: {method}")
                continue
            
            fig, axes = plt.subplots(1, 2, figsize=(16, 6))
            fig.suptitle(f"Method: {method} - Dataset-Level Rank (Lower is Better)", 
                        fontsize=18, fontweight='bold', y=1.02)
            
            # Helper for clean plotting
            def plot_clean(ax, x_col, label, log_scale=False):
                if x_col not in method_data.columns: 
                    ax.text(0.5, 0.5, f'No {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Remove NaN values for this specific plot
                plot_subset = method_data[[x_col, 'rank', 'dataset']].dropna()
                
                if len(plot_subset) == 0:
                    ax.text(0.5, 0.5, f'No valid {label} data', ha='center', va='center', 
                           transform=ax.transAxes, fontsize=12)
                    return
                
                # Scatter: Blue dots
                scatter = ax.scatter(
                    plot_subset[x_col], plot_subset['rank'],
                    color='tab:blue', s=80, alpha=0.7, edgecolor='darkblue', linewidth=1.5
                )
                
                # Trendline
                try:
                    sns.regplot(
                        data=plot_subset, x=x_col, y='rank',
                        ax=ax, scatter=False, color='red',
                        line_kws={'linestyle': '--', 'linewidth': 2, 'alpha': 0.7}
                    )
                except Exception as e:
                    print(f"  ⚠️  Could not fit trendline for {label}: {e}")
                
                # Formatting
                ax.set_xlabel(label, fontsize=12, fontweight='bold')
                ax.set_ylabel('Rank (Lower = Better)', fontsize=12, fontweight='bold')
                
                # Y-axis: Start from 0.5, go up to max rank + 0.5
                y_max = plot_data['rank'].max() + 0.5
                ax.set_ylim(0.5, y_max)
                
                # Log scale for x if requested
                if log_scale:
                    ax.set_xscale('log')
                
                # Grid
                ax.grid(True, linestyle=':', alpha=0.4, linewidth=0.8)

            # Plot 1: Rows (Left) - Log scale
            plot_clean(axes[0], 'n_rows', 'Number of Rows (Log Scale)', log_scale=True)
            axes[0].set_title("Rank vs Dataset Size", fontsize=13, pad=10, fontweight='bold')

            # Plot 2: Features (Middle) - Log scale
            plot_clean(axes[1], 'n_features', 'Number of Features (Log Scale)', log_scale=True)
            axes[1].set_title("Rank vs Feature Count", fontsize=13, pad=10, fontweight='bold')


            plt.tight_layout(rect=[0, 0, 1, 0.98])
            
            filename = FIGURES_DIR / f"lgd_rank_scatter_{method}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            print(f"\n✅ Saved: {filename.name}")
            plt.show()
            
        print("\n" + "="*80)
        print("✅ All scatter plots generated successfully!")
        print("  These plots now use EXACTLY the same ranking as the heatmap:")
        print("  - Ranks based on mean R² per dataset")
        print("  - Each dot = one dataset")
        print("  - Rank 1 dots should match heatmap exactly")
        print("="*80)

---
# ⏱️ Part D: Training Time Analysis

Analyze computational efficiency of methods:
- **Training time matrices**: Dataset × Method training times
- **Average times**: Method efficiency comparison
- **HPO overhead**: Additional cost of hyperparameter tuning

## D1. PD Training Time Analysis

In [ ]:
print("\n" + "=" * 80)
print("  PD - TRAINING TIME ANALYSIS (NO_HPO)")
print("=" * 80)

# Helper function to format time
def format_time(seconds):
    """Format time as minutes or seconds with appropriate suffix."""
    if seconds >= 60:
        minutes = seconds / 60
        return f"{minutes:.2f}m"
    else:
        return f"{seconds:.2f}s"

# Check if training time data is available in raw results
if 'train_time' in pd_raw.columns:
    # Calculate TOTAL training time per method-dataset (sum across all folds)
    time_summary = pd_raw.groupby(['dataset', 'method', 'hpo_mode'])['train_time'].sum().reset_index()
    time_summary.rename(columns={'train_time': 'total_train_time'}, inplace=True)
    
    # Filter for NO_HPO only
    df_no_hpo = time_summary[time_summary['hpo_mode'] == 'NO_HPO'].copy()
    
    if not df_no_hpo.empty:
        # =======================================================================
        # TRAINING TIME HEATMAP
        # =======================================================================
        print("\nCreating PD training time heatmap (NO_HPO)...")
        
        pivot_time = df_no_hpo.pivot(index='dataset', columns='method', values='total_train_time')
        pivot_time = pivot_time.sort_index()
        
        # Sort methods by average training time (fastest to slowest)
        method_means = pivot_time.mean(axis=0).sort_values()
        pivot_time = pivot_time[method_means.index]
        
        # Create formatted version for display
        pivot_time_formatted = pivot_time.applymap(format_time)
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            pivot_time, annot=pivot_time_formatted, fmt='', cmap='YlOrRd',
            cbar_kws={'label': 'Total Training Time (seconds)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('PD: Total Training Time per Dataset-Method (NO_HPO)\nDatasets × Methods (sum across all folds)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "pd_training_time_total_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # TRAINING TIME RANK MATRIX
        # =======================================================================
        print("\nCreating PD training time rank matrix (NO_HPO)...")
        
        # Calculate ranks (1 = fastest)
        rank_time = pivot_time.rank(axis=1, method='average')
        
        # Sort by average rank
        method_avg_ranks = rank_time.mean(axis=0).sort_values()
        rank_time = rank_time[method_avg_ranks.index]
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            rank_time, annot=True, fmt='.1f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Rank (1=fastest)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('PD: Training Time Ranks per Dataset-Method (NO_HPO)\nDatasets × Methods (1=fastest)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "pd_training_time_ranks_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # SUMMARY STATISTICS
        # =======================================================================
        print("\n" + "=" * 80)
        print("  TRAINING TIME SUMMARY (NO_HPO)")
        print("=" * 80)
        
        print("\nAverage Total Training Time per Method:")
        method_totals = df_no_hpo.groupby('method')['total_train_time'].mean().sort_values()
        for method, time_val in method_totals.items():
            print(f"  {method:20s}: {format_time(time_val):>10s}")
        
        print("\n" + "=" * 80)
        
    else:
        print("⚠️  No NO_HPO data available")
        
else:
    print("⚠️  Training time data not available in raw results")
    print("     Expected column: 'train_time'")
    print(f"     Available columns: {list(pd_raw.columns)}")

## D2. LGD Training Time Analysis

In [ ]:
print("\n" + "=" * 80)
print("  LGD - TRAINING TIME ANALYSIS (NO_HPO)")
print("=" * 80)

# Helper function to format time
def format_time(seconds):
    """Format time as minutes or seconds with appropriate suffix."""
    if seconds >= 60:
        minutes = seconds / 60
        return f"{minutes:.2f}m"
    else:
        return f"{seconds:.2f}s"

# Check if training time data is available in raw results
if 'train_time' in lgd_raw.columns:
    # Calculate TOTAL training time per method-dataset (sum across all folds)
    time_summary = lgd_raw.groupby(['dataset', 'method', 'hpo_mode'])['train_time'].sum().reset_index()
    time_summary.rename(columns={'train_time': 'total_train_time'}, inplace=True)
    
    # Filter for NO_HPO only
    df_no_hpo = time_summary[time_summary['hpo_mode'] == 'NO_HPO'].copy()
    
    if not df_no_hpo.empty:
        # =======================================================================
        # TRAINING TIME HEATMAP
        # =======================================================================
        print("\nCreating LGD training time heatmap (NO_HPO)...")
        
        pivot_time = df_no_hpo.pivot(index='dataset', columns='method', values='total_train_time')
        pivot_time = pivot_time.sort_index()
        
        # Sort methods by average training time (fastest to slowest)
        method_means = pivot_time.mean(axis=0).sort_values()
        pivot_time = pivot_time[method_means.index]
        
        # Create formatted version for display
        pivot_time_formatted = pivot_time.applymap(format_time)
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            pivot_time, annot=pivot_time_formatted, fmt='', cmap='YlOrRd',
            cbar_kws={'label': 'Total Training Time (seconds)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('LGD: Total Training Time per Dataset-Method (NO_HPO)\nDatasets × Methods (sum across all folds)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_training_time_total_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # TRAINING TIME RANK MATRIX
        # =======================================================================
        print("\nCreating LGD training time rank matrix (NO_HPO)...")
        
        # Calculate ranks (1 = fastest)
        rank_time = pivot_time.rank(axis=1, method='average')
        
        # Sort by average rank
        method_avg_ranks = rank_time.mean(axis=0).sort_values()
        rank_time = rank_time[method_avg_ranks.index]
        
        fig, ax = plt.subplots(figsize=(24, 12))
        
        sns.heatmap(
            rank_time, annot=True, fmt='.1f', cmap='RdYlGn_r',
            cbar_kws={'label': 'Rank (1=fastest)'}, linewidths=0.5, ax=ax
        )
        
        ax.set_title('LGD: Training Time Ranks per Dataset-Method (NO_HPO)\nDatasets × Methods (1=fastest)', 
                     fontsize=16, fontweight='bold', pad=20)
        ax.set_xlabel('Method', fontsize=12, fontweight='bold')
        ax.set_ylabel('Dataset', fontsize=12, fontweight='bold')
        
        plt.tight_layout()
        filename = FIGURES_DIR / "lgd_training_time_ranks_no_hpo.png"
        plt.savefig(filename, dpi=300, bbox_inches='tight')
        print(f"✅ Saved: {filename.name}")
        plt.show()
        
        # =======================================================================
        # SUMMARY STATISTICS
        # =======================================================================
        print("\n" + "=" * 80)
        print("  TRAINING TIME SUMMARY (NO_HPO)")
        print("=" * 80)
        
        print("\nAverage Total Training Time per Method:")
        method_totals = df_no_hpo.groupby('method')['total_train_time'].mean().sort_values()
        for method, time_val in method_totals.items():
            print(f"  {method:20s}: {format_time(time_val):>10s}")
        
        print("\n" + "=" * 80)
        
    else:
        print("⚠️  No NO_HPO data available")
        
else:
    print("⚠️  Training time data not available in raw results")
    print("     Expected column: 'train_time'")
    print(f"     Available columns: {list(lgd_raw.columns)}")

---
# Part E: Combined Statistical Analysis (PD + LGD)

Following Demsar (2006), we conduct a unified statistical analysis across both PD and LGD tasks.
- **Friedman test** for omnibus differences among methods
- **Pairwise paired T-tests** with Holm p-value correction
- **Critical Difference diagrams** visualising groups of statistically indistinguishable methods

Only methods present in both PD and LGD benchmarks are included in the combined analysis.

## E1. Combined Pairwise Comparisons & Critical Difference

In [ ]:
print("\n" + "=" * 80)
print("  COMBINED PD + LGD - STATISTICAL ANALYSIS (HPO)")
print("=" * 80)
print("  Following Demsar (2006) and the paper's Section 5.3 methodology:")
print("  - Method families: SVM+SVR and Linear+Logistic treated as one entity")
print("  - Performance ranks computed per (dataset, fold)")
print("  - Friedman test for omnibus differences")
print("  - Pairwise T-tests with Holm correction on standardised scores")
print("  - PAMA analysis & Critical Difference diagram")
print("=" * 80)

# Build performance matrices for both tasks
pd_perf_comb = compute_performance_matrix(pd_raw, 'AUC', hpo_mode='HPO')
lgd_perf_comb = compute_performance_matrix(lgd_raw, 'R2', hpo_mode='HPO')

if pd_perf_comb is not None and lgd_perf_comb is not None:
    # --- Method Family Grouping ---
    # Paper: "We treat SVM and SVR, as well as linear and logistic regression,
    #  as members of the same model family"
    pd_family = group_performance_by_family(pd_perf_comb)
    lgd_family = group_performance_by_family(lgd_perf_comb)

    print(f"\nPD methods: {len(pd_perf_comb.columns)} -> {len(pd_family.columns)} families")
    print(f"LGD methods: {len(lgd_perf_comb.columns)} -> {len(lgd_family.columns)} families")

    # Find families common to both tasks
    common_families = sorted(set(pd_family.columns) & set(lgd_family.columns))
    print(f"\nFamilies common to both PD and LGD: {len(common_families)}")
    for m in common_families:
        marker = " *" if m in FOUNDATION_METHODS else ""
        print(f"  {m}{marker}")

    pd_only_fam = sorted(set(pd_family.columns) - set(lgd_family.columns))
    lgd_only_fam = sorted(set(lgd_family.columns) - set(pd_family.columns))
    if pd_only_fam:
        print(f"\nPD-only families (excluded): {pd_only_fam}")
    if lgd_only_fam:
        print(f"LGD-only families (excluded): {lgd_only_fam}")

    if len(common_families) >= 3:
        pd_common = pd_family[common_families]
        lgd_common = lgd_family[common_families]

        n_pd = len(pd_common)
        n_lgd = len(lgd_common)
        n_total = n_pd + n_lgd

        # =================================================================
        # PAMA Analysis (Combined)
        # =================================================================
        print("\n" + "-" * 60)
        print("PAMA Analysis (Combined PD + LGD)")
        print("-" * 60)

        combined_perf_for_pama = pd.concat([pd_common, lgd_common], axis=0, ignore_index=True)
        combined_pama = compute_pama(combined_perf_for_pama, higher_is_better=True)

        print(f"\nCombined PAMA ({n_total} observations = {n_pd} PD + {n_lgd} LGD):")
        for method, frac in combined_pama.items():
            marker = " *" if method in FOUNDATION_METHODS else ""
            print(f"  {method:25s}  {frac*100:5.1f}%  ({int(frac*n_total):3d}/{n_total} wins){marker}")

        fm_wins = sum(frac for m, frac in combined_pama.items() if m in FOUNDATION_METHODS)
        print(f"\n  Foundation models (group): {fm_wins*100:.1f}%")

        plot_pama(
            combined_pama,
            title=f'Combined PD+LGD: PAMA Analysis (HPO)\n{n_total} obs ({n_pd} PD + {n_lgd} LGD), {len(common_families)} families',
            foundation_methods=FOUNDATION_METHODS,
            figsize=(16, max(6, len(common_families) * 0.45)),
            save_path=FIGURES_DIR / "combined_pama_hpo.png"
        )

        # =================================================================
        # Rank-based analysis (Friedman)
        # =================================================================
        pd_ranks = pd_common.rank(axis=1, ascending=False, method='average')
        lgd_ranks = lgd_common.rank(axis=1, ascending=False, method='average')
        combined_ranks = pd.concat([pd_ranks, lgd_ranks], axis=0, ignore_index=True)
        combined_avg_ranks = combined_ranks.mean().sort_values()

        print(f"\nCombined rank matrix: {n_total} observations x {len(common_families)} families")

        # Friedman test
        f_stat_comb, f_p_comb = friedmanchisquare(
            *[combined_ranks.iloc[:, i] for i in range(combined_ranks.shape[1])]
        )
        sep = "\u2500" * 60
        print(f"\n{sep}")
        print(f"Friedman test (combined):  \u03c7\u00b2 = {f_stat_comb:.2f},  p = {f_p_comb:.2e}")
        if f_p_comb < 0.05:
            print("  -> Significant differences exist among methods (p < 0.05)")
        print(sep)

        print(f"\nCombined Average Ranks (1 = best):")
        for method, rank in combined_avg_ranks.items():
            marker = " *" if method in FOUNDATION_METHODS else ""
            print(f"  {method:25s}  {rank:.2f}{marker}")

        # =================================================================
        # Pairwise T-tests on standardised scores
        # =================================================================
        pd_z = pd_common.subtract(pd_common.mean(axis=1), axis=0).divide(
            pd_common.std(axis=1).replace(0, 1), axis=0)
        lgd_z = lgd_common.subtract(lgd_common.mean(axis=1), axis=0).divide(
            lgd_common.std(axis=1).replace(0, 1), axis=0)
        pd_z = pd_z.replace([np.inf, -np.inf], np.nan).fillna(0)
        lgd_z = lgd_z.replace([np.inf, -np.inf], np.nan).fillna(0)
        combined_z = pd.concat([pd_z, lgd_z], axis=0, ignore_index=True)

        combined_pairwise = pairwise_ttest_holm(combined_z)
        n_sig = combined_pairwise['significant'].sum()
        n_total_pairs = len(combined_pairwise)
        print(f"\nPairwise T-tests on standardised scores (Holm correction):")
        print(f"  {n_sig}/{n_total_pairs} pairs significantly different (alpha=0.05)")

        sig_pairs = combined_pairwise[combined_pairwise['significant']].sort_values('p_corrected')
        if len(sig_pairs) > 0:
            print(f"\n  Top significant pairs:")
            for _, row in sig_pairs.head(20).iterrows():
                print(f"    {row['method_a']:25s} vs {row['method_b']:25s}  p={row['p_corrected']:.4f}")

        nonsig = combined_pairwise[~combined_pairwise['significant']]
        if len(nonsig) > 0:
            print(f"\n  Non-significant pairs ({len(nonsig)}):")
            for _, row in nonsig.sort_values('p_corrected', ascending=False).head(20).iterrows():
                print(f"    {row['method_a']:25s} vs {row['method_b']:25s}  p={row['p_corrected']:.4f}")

        # =================================================================
        # Critical Difference Diagram
        # =================================================================
        print("\n" + "-" * 60)
        print("Generating Combined Critical Difference diagram...")
        plot_critical_difference_diagram(
            combined_avg_ranks, combined_pairwise,
            title=(f'Combined PD+LGD: Critical Difference Diagram (HPO)\n'
                   f'Friedman p={f_p_comb:.2e}  |  '
                   f'{n_total} obs ({n_pd} PD + {n_lgd} LGD)  |  {len(common_families)} families'),
            figsize=(20, max(10, len(combined_avg_ranks) * 0.5)),
            foundation_methods=FOUNDATION_METHODS,
            save_path=FIGURES_DIR / "combined_critical_difference_hpo.png"
        )

        # Pairwise significance heatmap
        print("Generating Combined pairwise significance heatmap...")
        plot_pairwise_significance_heatmap(
            combined_pairwise, combined_avg_ranks,
            title='Combined PD+LGD: Pairwise T-Test p-values (Holm-corrected, HPO)',
            save_path=FIGURES_DIR / "combined_pairwise_significance_heatmap_hpo.png"
        )
    else:
        print(f"Warning: Only {len(common_families)} families common to PD and LGD -- need >= 3")
else:
    print("Warning: Could not build performance matrices for both PD and LGD")

---
## Summary

This notebook analyzed Experiment1 results with comprehensive visualizations:

**For both PD and LGD tasks:**
- Performance heatmaps (NO_HPO, HPO, improvement)
- Average performance bar charts with error bars
- Performance distribution boxplots across folds
- Rank heatmaps and average rank bar charts
- PAMA analysis at fold level (not dataset level)
- Rank correlation with dataset characteristics (size, features, dimensionality)
- **Training time analysis** (heatmaps, averages)
- **Statistical significance testing** (Friedman test, pairwise T-tests with Holm correction, Critical Difference diagrams)

**Combined PD + LGD statistical analysis:**
- Friedman omnibus test for overall method differences
- Pairwise paired T-tests with Holm p-value correction on standardised scores
- Critical Difference diagrams (Demsar, 2006) with non-significant cliques
- Pairwise significance heatmaps

All visualizations saved to `figures/` directory at 300 DPI.